# Hybrid (HB) Segment Raporu

Aylık müşteri portföyü dosyalarınızdan (`202603.csv … 202608.csv`) **üç sayfalık, sol menülü, tek dosyalık HTML rapor** üretir.

1. **Ana Ekran:** son ayın güncel durumu (karşılaştırmasız): ARPU, profil, kullanım, tarife / cihaz / dijital segment, dijital ilgi alanları, risk sinyalleri.
2. **Karşılaştırma:** iki ayı seçip farkları görün; sayfanın altında adet ve ortalamalardaki ciddi değişimler otomatik raporlanır (eşikler ayarlanabilir).
3. **Müşteri Hareketleri:** şirket içi / dışı gelen ve gidenler; hareket türüne ve kırılımlara göre profil (ARPU, internet kullanımı, yaş, kıdem, ilgi alanı vb.).

## Kendi verinizle çalıştırmak
1. CSV dosyalarını bir klasöre koyun; adları `YYYYMM.csv` olmalı (örn. `202603.csv`).
2. **Ayarlar** hücresinde `VERI_KLASORU = "klasör_yolu"` yazın.
3. **Run All.** İlk çıktıyı "veri doğrulama" bölümünde kontrol edin: dosya biçimi (kodlama, ayraç, ondalık otomatik algılanır), eksik kolonlar, bulunan cinsiyet ve cihaz kolonları, ay bazında hareket adetleri.

Cinsiyet ve cihaz markası kolonlarının adı şemada olmadığı için `GENDER, CINSIYET, SEX …` ve `DEVICE_BRAND, HANDSET_BRAND, BRAND …` adayları otomatik denenir. Adınız farklıysa `CINSIYET_KOLONU` / `CIHAZ_KOLONU` ayarına yazın. Bu kolonlar yoksa ilgili analizler raporda hiç görünmez, rapor yine çalışır.

`PHONE` kolonu okunmaz, rapora girmez.

## Hareket kuralları (doğrulayın)
- Aktif durumdaki ve çıkış bayrağı olmayan satır **portföyde** sayılır.
- `HB2PRE=1` → Prepaid'e, `HB2POST=1` → Postpaid'e giden. Pasif durum + `POUT_COUNT_LM>0` → rakibe (port-out). Diğer pasif/iptal → hat kapatma.
- Giriş: `HBPORTIN` numara taşıma, `HBACQ` yeni satış, `PRE2HB`/`POST2HB` diğer segmentten; bayrağı olmayan yeni müşteri "diğer giriş".
- Önceki ayın portföyünde olup bu ay **hiç satırı olmayan** müşteri "satırı kaybolan (sınıflanamayan çıkış)" olarak ayrıca gösterilir.
Kurallar `durum_kodu()` ve `giris_kodu()` fonksiyonlarında tek yerdedir.

### 1 · Ayarlar

In [ ]:
import json, math, re, os, csv, io, gzip, base64, datetime as dt
import numpy as np
import pandas as pd

# ═══════════════════════════ AYARLAR ═══════════════════════════
SEGMENT_ADI    = "Hybrid (HB)"        # Başlıkta görünür
CIKTI_HTML     = "hybrid_rapor.html"
VERI_KLASORU   = "veri"               # içinde 202603.csv, 202604.csv … 202608.csv bulunan klasör (yol)
DOSYA_DESENI   = r"^(\d{6})\.csv$"    # ay dosyası adı deseni (YYYYMM.csv)
PLOTLY_GOMULU  = True                 # True: internetsiz açılır (+~4.5 MB)
VERI_SIKISTIR  = True                 # True: veri gzip+base64 (Chrome/Edge 80+) | False: ham JSON
NOTEBOOKTA_GOSTER = False
TANILAMA       = False                # True: hata olmasa da dosya yapısı özetini (kişisel veri içermez) basar

# ── Kolon eşlemeleri (gerçek veride kontrol edin; yanlışsa buradan düzeltin) ──
ANAHTAR_KOLONU = None                 # müşteriyi aylar arası takip eden kimlik kolonu. None → aşağıdaki adaylardan benzersizlik ve aylar arası örtüşmeye göre otomatik seçilir
ANAHTAR_ADAYLARI = ["CAMPAIGN_SUBSCRIBER_ID", "SUBSCRIBER_ID", "PHONE"]   # PHONE seçilirse yalnızca hash'lenerek kullanılır, hiçbir yerde saklanmaz/gösterilmez
ANAHTAR        = "MUSTERI_ANAHTAR"    # (dahili ad, değiştirmeyin)
KIRILIM_DISI   = ["tar", "con", "chg"]   # segmentte sabit olan kolonlar (TARIFF_ID, CONTRACT_TYPE_ID, CHARGING_METHOD_ID) kırılım olarak kullanılmaz
KAMPANYA_KOLON = "CAMPAIGN_SUBSCRIBER_ID"
YAS_KOLONU     = "SUBS_AGE"           # kişi yaşı (yıl) olmalı
AKTIF_DURUM_IDS = None                # SUBSCRIBER_STATUS_ID sabitse (tek değer) tüm satırlar aktif sayılır. Birden fazla değer varsa: SUBSCRIBER_STATUS_ID içinde "aktif" sayılan değer(ler), örn. [2]. None → son aydaki en sık değer otomatik seçilir (ekrana yazılır)
BONUS_ADAYLARI = ["BONUS_NAME"]                                 # bonus adı kolonu (yoksa bonus kırılımı çıkmaz)
TARIFE_ADI_ADAYLARI = ["OPTION_AND_PACKAGE_NAME", "TARIFF_NAME", "TARIFF_DESC"]   # TARIFF_ID etiketi için tarife adı kolonu (ilk bulunan; en sık ad kullanılır)
SOZLESME_ISIM  = {1: "Taahhütlü", 2: "Taahhütsüz"}
UCRET_ISIM     = {1: "Ücretlendirme 1", 2: "Ücretlendirme 2", 3: "Ücretlendirme 3"}
TARIFE_ISIM    = {}                   # {TARIFF_ID: "Hybrid 40 GB", ...}

# Şemada olmayan ama sizde bulunan kolonlar: önce adayları sırayla dener, bulamazsa ilgili analiz atlanır.
# Kolon adınız farklıysa kolon adını doğrudan yazın (örn. CINSIYET_KOLONU = "GENDER_CD").
CINSIYET_KOLONU = None                # None → aşağıdaki adaylardan otomatik
CINSIYET_ADAYLARI = ["GENDER", "CINSIYET", "SEX", "GENDER_CD", "GENDER_ID", "GENDER_NAME"]
CINSIYET_ESLEME = {"K": "Kadın", "KADIN": "Kadın", "F": "Kadın", "FEMALE": "Kadın", "WOMAN": "Kadın",
                   "E": "Erkek", "ERKEK": "Erkek", "M": "Erkek", "MALE": "Erkek", "MAN": "Erkek"}
CIHAZ_KOLONU   = None                 # None → aşağıdaki adaylardan otomatik
CIHAZ_ADAYLARI = ["DEVICE_BRAND", "DEVICE", "DEVICE_NAME", "HANDSET", "HANDSET_BRAND", "PHONE_BRAND", "DEVICE_MAKE", "DEVICE_MANUFACTURER", "BRAND", "CIHAZ_MARKA", "MARKA", "DEVICE_VENDOR"]

# ── Uyarı (bilgilendirme) varsayılanları — arayüzden de değiştirilebilir ──
UYARI_ADET_ESIK_PCT  = 3.0            # grup adedi değişimi (%)
UYARI_ORT_ESIK_PCT   = 5.0            # grup ortalaması değişimi (%)
UYARI_MIN_GRUP       = 300            # bu adetin altındaki gruplar uyarıda yok sayılır

### 2 · CSV yükleyici
Ayraç, kodlama (UTF-8 / cp1254), ondalık işareti ve kolon adı farklılıklarını (büyük-küçük harf, boşluk, bayrak kolonlarındaki ay adı) otomatik düzeltir.

In [ ]:
# ═══════════ VERİ YÜKLEME: CSV'ler → standart tablo (ayraç/kodlama/ondalık otomatik algılanır) ═══════════
import codecs, difflib

BAYRAKLAR = ['HBACQ', 'HBPORTIN', 'PRE2HB', 'HB2PRE', 'POST2HB', 'HB2POST', 'PRE2POST', 'POST2PRE']
BAYRAK_RE = re.compile(r'^(%s)_\d{6}_FL$' % '|'.join(BAYRAKLAR))
METIN_KOLONLAR = ['CITY', 'PHONE', 'BONUS_NAME', 'RAHAT_SEGMENT', 'DIGITAL_SEGMENT_NAME']
ID_KOLONLAR = ['TARIFF_ID', 'SUBSCRIBER_STATUS_ID', 'CONTRACT_TYPE_ID', 'CHARGING_METHOD_ID']
SAYISAL = (['SUBSCRIBERS_LAST_TENURE_MONTHS', 'SUBS_AGE', 'TARIFF_OPTION_PRICE', 'TARIFF_OPTION_VOICE_USED_AMNT', 'TARIFF_OPTION_DATA_USED_AMNT', 'ARPU_TL'] + [f'ARPU_L{i}M' for i in range(1, 7)] +
           ['PRE_MNP', 'PIN_COUNT_HIST', 'POUT_COUNT_HIST'] + [f'{p}_COUNT_{w}' for p in ('PIN', 'POUT') for w in ('LM', 'L3M', 'L6M', 'L12M', 'L24M')] +
           [f'{b}_FL' for b in BAYRAKLAR] + ['COMPLAINT_FL', 'TOT_VOI_MOU', 'TOT_DATA_VOL_GB'])
BEKLENEN = (['CAMPAIGN_SUBSCRIBER_ID', 'SUBSCRIBER_ID', 'MONTHLY_CALENDAR_ID'] + METIN_KOLONLAR + ID_KOLONLAR + SAYISAL)


def kolon_norm(c):
    n = re.sub(r'\s+', '_', str(c).replace('﻿', '').strip()).upper()
    return BAYRAK_RE.sub(lambda m: m.group(1) + '_FL', n) if BAYRAK_RE.match(n) else n


def dosyalari_bul(klasor):
    if not os.path.isdir(klasor):
        raise FileNotFoundError(f"Veri klasörü bulunamadı: '{klasor}'. VERI_KLASORU ayarını kontrol edin (şu an: {os.path.abspath(klasor)}).")
    pat = re.compile(DOSYA_DESENI); out = []
    for f in sorted(os.listdir(klasor)):
        m = pat.match(f)
        if m:
            out.append((int(m.group(1)), os.path.join(klasor, f)))
    if len(out) < 2:
        raise FileNotFoundError(f"'{klasor}' içinde YYYYMM.csv biçiminde en az 2 dosya bekleniyor; bulunan: {[os.path.basename(p) for _, p in out]}")
    return out


def _kodlama(yol):
    raw = open(yol, 'rb').read(2_000_000)
    for enc in ('utf-8-sig', 'utf-8', 'cp1254', 'latin-1'):
        try:
            codecs.getincrementaldecoder(enc)().decode(raw, final=False); return enc
        except UnicodeDecodeError:
            continue
    return 'latin-1'


def _ayrac(yol, enc):
    with open(yol, 'r', encoding=enc, errors='replace') as f:
        satirlar = [f.readline() for _ in range(5)]
    sayim = {s: sum(l.count(s) for l in satirlar) for s in [',', ';', '\t', '|']}
    return max(sayim, key=sayim.get)


def _ondalik(yol, enc, sep):
    ornek = pd.read_csv(yol, sep=sep, encoding=enc, nrows=3000, dtype=str, usecols=lambda c: kolon_norm(c) in ('ARPU_TL', 'TOT_DATA_VOL_GB', 'TARIFF_OPTION_PRICE'))
    dec, th = '.', None
    for c in ornek.columns:
        s = ornek[c].dropna().astype(str)
        if len(s) and sep != ',' and s.str.match(r'^\s*-?\d{1,3}(\.\d{3})*,\d+\s*$').mean() > .2:
            dec = ','
            th = '.' if s.str.contains(r'^\s*-?\d{1,3}\.\d{3},').mean() > .01 else None
    return dec, th


def csv_oku(yol, mid):
    """Tek bir ay dosyasını okur, kolon adlarını standartlaştırır, tipleri düzeltir. (df, bilgi) döndürür."""
    enc = _kodlama(yol); sep = _ayrac(yol, enc); dec, th = _ondalik(yol, enc, sep)
    baslik = list(pd.read_csv(yol, sep=sep, encoding=enc, nrows=0).columns); norm = {c: kolon_norm(c) for c in baslik}
    ozel = [x for x in (CINSIYET_KOLONU, CIHAZ_KOLONU) if x]
    ister = set(BEKLENEN) | set(ANAHTAR_ADAYLARI) | set(BONUS_ADAYLARI) | set(TARIFE_ADI_ADAYLARI) | set(CINSIYET_ADAYLARI) | set(CIHAZ_ADAYLARI) | {kolon_norm(x) for x in ozel} | {KAMPANYA_KOLON, ANAHTAR, YAS_KOLONU}
    kullan = [c for c in baslik if norm[c] in ister or norm[c].endswith('_SEARCH')]
    df = pd.read_csv(yol, sep=sep, encoding=enc, decimal=dec, thousands=th, usecols=kullan, low_memory=False)
    df.columns = [norm[c] for c in df.columns]
    df = df.loc[:, ~df.columns.duplicated()]
    kaynak = globals().get('ANAHTAR_KAYNAK') or ANAHTAR_KOLONU or 'SUBSCRIBER_ID'
    if kaynak not in df.columns:
        yakin = difflib.get_close_matches(kaynak, list(norm.values()), n=3)
        raise KeyError(f"{os.path.basename(yol)}: anahtar kolon '{kaynak}' bulunamadı. Benzer kolonlar: {yakin}. ANAHTAR_KOLONU ayarını düzeltin.")
    df[ANAHTAR] = pd.util.hash_array(df[kaynak].astype(str).to_numpy()) if kaynak == 'PHONE' else df[kaynak]
    bilgi = dict(ay=mid, dosya=os.path.basename(yol), kodlama=enc, ayrac=repr(sep), ondalik=dec, satir=len(df), kolon=len(baslik))
    n0 = len(df); bilgi['ham_satir'] = n0; bilgi['anahtar_kolonu'] = kaynak; bilgi['benzersiz_anahtar'] = int(df[ANAHTAR].nunique())
    bilgi['benzersiz_kampanya'] = int(df[KAMPANYA_KOLON].nunique()) if KAMPANYA_KOLON in df.columns else None
    bilgi['benzersiz_subscriber'] = int(df['SUBSCRIBER_ID'].nunique()) if 'SUBSCRIBER_ID' in df.columns else None
    df = df.drop_duplicates(subset=[ANAHTAR], keep='first'); bilgi['tekrar_eden_id'] = n0 - len(df)
    # sayısal tipler
    for c in SAYISAL:
        if c in df.columns:
            num = pd.to_numeric(df[c], errors='coerce'); df[c] = num.astype('float32') if num.dtype == 'float64' and not c.endswith('_FL') else num
    for c in ID_KOLONLAR:
        if c in df.columns:
            num = pd.to_numeric(df[c], errors='coerce')
            df[c] = num if num.notna().mean() > .9 else df[c].astype(str).str.strip()
    for c in METIN_KOLONLAR:
        if c in df.columns and c != 'PHONE':
            df[c] = df[c].astype('object').where(df[c].notna(), None); df[c] = df[c].map(lambda v: v.strip() if isinstance(v, str) else v)
    if 'PHONE' in df.columns:
        df = df.drop(columns=['PHONE'])                               # kişisel veri: rapora hiç alınmaz
    # cinsiyet / cihaz markası
    gk = kolon_norm(CINSIYET_KOLONU) if CINSIYET_KOLONU else next((c for c in CINSIYET_ADAYLARI if c in df.columns), None)
    if gk and gk in df.columns:
        v = df[gk].astype(str).str.strip().str.upper(); std = v.map(CINSIYET_ESLEME)
        df['CINSIYET_STD'] = std.where(std.notna(), pd.Series(np.where(v.isin(['NAN', '', 'NONE']), None, 'Cinsiyet ' + v), index=v.index))
    bilgi['cinsiyet_kolonu'] = gk if gk in df.columns else None
    ck = kolon_norm(CIHAZ_KOLONU) if CIHAZ_KOLONU else next((c for c in CIHAZ_ADAYLARI if c in df.columns), None)
    if ck and ck in df.columns:
        df['CIHAZ_STD'] = df[ck].astype(str).str.strip().str.upper().replace({'NAN': None, '': None, 'NONE': None})
    bilgi['cihaz_kolonu'] = ck if ck in df.columns else None
    if 'BONUS_NAME' not in df.columns:
        bk = next((c for c in BONUS_ADAYLARI if c in df.columns), None)
        if bk: df['BONUS_NAME'] = df[bk]
    tk = next((c for c in TARIFE_ADI_ADAYLARI if c in df.columns), None)
    if tk: df['TARIFE_ADI_STD'] = df[tk].astype(str).str.strip().where(df[tk].notna(), None)
    bilgi['tarife_adi_kolonu'] = tk
    # eksik beklenen kolonlar: pipeline çökmesin diye boş eklenir, raporda uyarı verilir
    eksik = [c for c in BEKLENEN if c not in df.columns and c != 'PHONE']
    for c in eksik:
        df[c] = np.nan if c in SAYISAL else None
    if 'SUBSCRIBER_STATUS_ID' in eksik: df['SUBSCRIBER_STATUS_ID'] = 1
    for b in BAYRAKLAR:
        if f'{b}_FL' in eksik: df[f'{b}_FL'] = 0
    df[[f'{b}_FL' for b in BAYRAKLAR]] = df[[f'{b}_FL' for b in BAYRAKLAR]].fillna(0)
    bilgi['eksik_kolonlar'] = [c for c in eksik if c != 'CAMPAIGN_SUBSCRIBER_ID']
    bilgi['arama_kolonu'] = sum(c.endswith('_SEARCH') for c in df.columns)
    if 'MONTHLY_CALENDAR_ID' in df.columns and df['MONTHLY_CALENDAR_ID'].notna().any():
        farkli = pd.to_numeric(df['MONTHLY_CALENDAR_ID'], errors='coerce').dropna().astype(int).unique()
        bilgi['ay_kolonu_uyumsuz'] = bool(len(farkli) and (farkli != mid).any())
    return df, bilgi


def anahtar_sec(dosyalar):
    """Müşteri anahtarını seçer: ay içinde benzersiz (≥%98) ve önceki ayla en çok örtüşen aday. ANAHTAR_KOLONU verilmişse onu kullanır."""
    g = globals()
    if ANAHTAR_KOLONU:
        g['ANAHTAR_KAYNAK'] = kolon_norm(ANAHTAR_KOLONU); print(f"→ Müşteri anahtarı (ayardan): {g['ANAHTAR_KAYNAK']}"); return
    (m0, y0), (m1, y1) = dosyalar[-2], dosyalar[-1]; veri = {}
    for m, y in [(m0, y0), (m1, y1)]:
        enc = _kodlama(y); sep = _ayrac(y, enc)
        hdr = [c for c in pd.read_csv(y, sep=sep, encoding=enc, nrows=0).columns if kolon_norm(c) in set(ANAHTAR_ADAYLARI)]
        d = pd.read_csv(y, sep=sep, encoding=enc, usecols=hdr, low_memory=False); d.columns = [kolon_norm(c) for c in d.columns]; veri[m] = d
    satir = []
    for a in ANAHTAR_ADAYLARI:
        if a not in veri[m1].columns: continue
        k1, k0 = veri[m1][a], veri[m0][a]
        if a == 'PHONE': k1, k0 = pd.Series(pd.util.hash_array(k1.astype(str).to_numpy())), pd.Series(pd.util.hash_array(k0.astype(str).to_numpy()))
        u = k1.nunique(); satir.append(dict(aday=a, ham_satir=len(k1), benzersiz=u, benzersiz_oran=u / max(len(k1), 1), onceki_ayda_bulunan=float(k1.drop_duplicates().isin(set(k0.dropna())).mean())))
    t = pd.DataFrame(satir)
    if t.empty: raise KeyError(f"Anahtar adaylarından hiçbiri dosyada yok: {ANAHTAR_ADAYLARI}. ANAHTAR_KOLONU ayarını yazın.")
    print("── Müşteri anahtarı adayları (son ay) ──"); print(t.assign(benzersiz_oran=(100 * t.benzersiz_oran).round(1), onceki_ayda_bulunan=(100 * t.onceki_ayda_bulunan).round(1)).rename(columns={'benzersiz_oran': 'benzersiz_%', 'onceki_ayda_bulunan': 'önceki_ayda_bulunan_%'}).to_string(index=False))
    uygun = t[t.benzersiz_oran >= .98]
    sec = (uygun if len(uygun) else t).sort_values(['onceki_ayda_bulunan', 'benzersiz_oran'], ascending=False).iloc[0]['aday']
    g['ANAHTAR_KAYNAK'] = sec; print(f"→ Müşteri anahtarı OTOMATİK seçildi: {sec}. Yanlışsa ayarlardaki ANAHTAR_KOLONU'na doğru kolonu yazın." + ("" if len(uygun) else "  ⚠ hiçbir aday ay içinde tam benzersiz değil!"))

### 3 · Dönüşüm hattı
Aylık portföylerden rapor tablolarını üretir; hareket kuralları burada.

In [ ]:
# ═══════════ DÖNÜŞÜM HATTI: aylık portföy tabloları → rapor tabloları ═══════════
ILGI_ETIKET = {'BANK': 'Bankacılık', 'SOCIALMEDIA': 'Sosyal medya', 'NEWS': 'Haber', 'SPORT': 'Spor', 'GAME': 'Oyun', 'MUSIC': 'Müzik', 'STREAMING': 'Video yayın', 'ECOMMERCE': 'E-ticaret',
               'HEALTH': 'Sağlık', 'RETAIL': 'Perakende', 'CLOTHES': 'Giyim', 'ACTUERIA': 'Sigorta', 'MAKEUP': 'Kozmetik', 'LUXURY': 'Lüks', 'FOODORDER': 'Yemek siparişi', 'ART': 'Sanat',
               'CHILDCARE': 'Çocuk bakımı', 'GAMBLING': 'Şans oyunları', 'CARRENT': 'Araç kiralama', 'CARBUY': 'Araç alım-satım', 'TECH': 'Teknoloji', 'SECONDHAND': 'İkinci el', 'ESTATE': 'Emlak',
               'TRAFFIC': 'Trafik / harita', 'TRAVEL': 'Seyahat'}
EV = ['stock', 'in_acq', 'in_portin', 'in_pre', 'in_post', 'in_oth', 'out_pre', 'out_post', 'out_rival', 'out_close', 'out_ghost']
EV_LBL = ['Portföy (segmentte olan)', 'Yeni satış', 'Numara taşıma ile gelen', "Prepaid'den gelen", "Postpaid'den gelen", 'Diğer giriş (sınıflanamayan)',
          "Prepaid'e giden", "Postpaid'e giden", 'Rakibe giden (port-out)', 'Hat kapatan / pasif', 'Dosyadan kaybolan (şirket dışı / sınıflanamayan çıkış)']
METR = [('arpu', 'ARPU', '₺'), ('gb', 'Data kullanımı', 'GB'), ('mou', 'Ses kullanımı', 'dk'), ('price', 'Tarife ücreti', '₺'), ('ten', 'Kıdem', 'ay'), ('age', 'Yaş', 'yıl'),
        ('arpur', 'ARPU değişimi (son 3 ay ort.na göre)', '%'), ('comp', 'Şikayet oranı', '%'), ('po3', 'Son 3 ayda port-out olan', '%'), ('pi12', 'Son 12 ayda port-in olan', '%')]
NMET = len(METR)
TRANS_DIMS = ['tar', 'dig', 'dev', 'con', 'city']
CROSS = [('age', 'gen'), ('age', 'tar'), ('tar', 'con'), ('dig', 'dev'), ('city', 'tar')]
HIST_EDGES = [-1.0, -.5, -.3, -.2, -.1, -.05, .05, .1, .2, .3, .5, 1.0, 1e9]


def _is_aktif(df):
    if globals().get('STATUS_SABIT'): return np.ones(len(df), bool)
    s = df['SUBSCRIBER_STATUS_ID']; num = pd.to_numeric(s, errors='coerce')
    return (num.isin(AKTIF_DURUM_IDS) if num.notna().mean() > .9 else s.astype(str).isin([str(x) for x in AKTIF_DURUM_IDS])).to_numpy()


def aktif_durum_belirle(df):
    """AKTIF_DURUM_IDS verilmediyse son aydaki en sık durum kimliğini aktif kabul eder ve ekrana yazar."""
    g = globals(); vc = df['SUBSCRIBER_STATUS_ID'].value_counts(dropna=False)
    if df['SUBSCRIBER_STATUS_ID'].nunique(dropna=True) <= 1:
        g['STATUS_SABIT'] = True; print("SUBSCRIBER_STATUS_ID sabit (tek değer) → tüm satırlar aktif müşteri sayılıyor."); return
    g['STATUS_SABIT'] = False; vc = df['SUBSCRIBER_STATUS_ID'].value_counts(dropna=True)
    print("SUBSCRIBER_STATUS_ID dağılımı (son ay):", {k: f"{v:,} (%{100*v/len(df):.1f})" for k, v in vc.head(8).items()})
    if g.get('AKTIF_DURUM_IDS') is None:
        g['AKTIF_DURUM_IDS'] = [vc.index[0].item() if hasattr(vc.index[0], 'item') else vc.index[0]]
        print(f"→ Aktif durum kimliği OTOMATİK seçildi: {g['AKTIF_DURUM_IDS']} (en sık değer). Yanlışsa ayarlardaki AKTIF_DURUM_IDS'i yazın.")
    else:
        print("→ Aktif durum kimliği (ayardan):", g['AKTIF_DURUM_IDS'])
    if not _is_aktif(df).any():
        raise ValueError(f"SUBSCRIBER_STATUS_ID içinde AKTIF_DURUM_IDS={g['AKTIF_DURUM_IDS']} ile eşleşen hiç satır yok. Dağılım: {dict(vc.head(8))}. Ayarlardaki AKTIF_DURUM_IDS'i bu değerlerden doğru olanla düzeltin.")


def durum_kodu(df):
    """HAREKET KURALI (gerekirse kendi veri sözleşmenize göre düzenleyin).
    0 portföyde · 1 Prepaid'e geçti (HB2PRE) · 2 Postpaid'e geçti (HB2POST) · 3 rakibe (pasif + port-out) · 4 hat kapatma/pasif"""
    aktif = _is_aktif(df); d = np.zeros(len(df), np.int8)
    d[~aktif] = 4; d[(~aktif) & (df['POUT_COUNT_LM'].fillna(0).to_numpy() > 0)] = 3
    d[df['HB2POST_FL'].fillna(0).to_numpy() == 1] = 2; d[df['HB2PRE_FL'].fillna(0).to_numpy() == 1] = 1
    return d


def giris_kodu(df):
    """Portföye giriş türü: 1 yeni satış · 2 numara taşıma · 3 PRE2HB · 4 POST2HB · 5 sınıflanamayan"""
    g = np.full(len(df), 5, np.int8)
    g[df['POST2HB_FL'].fillna(0).to_numpy() == 1] = 4; g[df['PRE2HB_FL'].fillna(0).to_numpy() == 1] = 3
    g[df['HBACQ_FL'].fillna(0).to_numpy() == 1] = 1; g[df['HBPORTIN_FL'].fillna(0).to_numpy() == 1] = 2
    return g


def _top(df, col, n, tr=str):
    v = df[col].dropna(); vc = v.value_counts(); return list(vc.index[:n])


def kat_dim(key, name, col, values, labfn, other=None, series=None, other_label='Diğer'):
    """Kategorik kırılım. 'other' None ise: seride listedeki değerler dışında (veya boş) değer varsa 'Diğer/Belirsiz' kategorisi eklenir."""
    mp = {v: i for i, v in enumerate(values)}
    if other is None:
        other = bool(series is not None and (series.isna().any() or series.nunique() > len(values)))
    labels = [labfn(v) for v in values] + ([other_label] if other else [])
    return dict(key=key, name=name, labels=labels, fn=lambda d, col=col, mp=mp, o=len(values), ot=other: d[col].map(mp).fillna(o if ot else 0).to_numpy().astype(np.uint8))


def _kenarlar(x, q=(.2, .4, .6, .8)):
    e = np.unique(np.round(np.nanquantile(x, q), 0)); return e


def _etiket_bant(e, birim=''):
    if len(e) == 0: return ['Tümü']
    L = [f'<{e[0]:.0f}{birim}'] + [f'{e[i]:.0f}–{e[i+1]:.0f}{birim}' for i in range(len(e) - 1)] + [f'{e[-1]:.0f}+{birim}']; return L


def dims_tanimla(df):
    s = df[_is_aktif(df)] if _is_aktif(df).any() else df
    D = []
    if 'CINSIYET_STD' in df and s['CINSIYET_STD'].notna().any():
        vals = [v for v in ['Kadın', 'Erkek'] if v in set(s['CINSIYET_STD'].dropna())] + sorted(set(s['CINSIYET_STD'].dropna()) - {'Kadın', 'Erkek'})
        D.append(kat_dim('gen', 'Cinsiyet', 'CINSIYET_STD', vals[:4], str, series=s['CINSIYET_STD'], other_label='Belirsiz'))
    if s[YAS_KOLONU].notna().any() and s[YAS_KOLONU].median() > 12:
        D.append(dict(key='age', name='Yaş', labels=['<25', '25-34', '35-44', '45-54', '55+'], fn=lambda d: np.digitize(d[YAS_KOLONU].fillna(-1).to_numpy(), [25, 35, 45, 55]).astype(np.uint8)))
    D.append(kat_dim('city', 'Şehir', 'CITY', _top(s, 'CITY', 10), str, series=s['CITY']))
    tmap = dict(TARIFE_ISIM)
    if not tmap and 'TARIFE_ADI_STD' in s and s['TARIFE_ADI_STD'].notna().any():            # tarife adı kolonundan TARIFF_ID etiketi: en sık ad
        tmap = s.dropna(subset=['TARIFE_ADI_STD']).groupby('TARIFF_ID')['TARIFE_ADI_STD'].agg(lambda x: x.value_counts().index[0]).to_dict()
    D.append(kat_dim('tar', 'Tarife', 'TARIFF_ID', _top(s, 'TARIFF_ID', 8), lambda v: tmap.get(v, f'Tarife {v:g}' if isinstance(v, (int, float, np.integer, np.floating)) else f'Tarife {v}'), series=s['TARIFF_ID']))
    D.append(kat_dim('con', 'Sözleşme', 'CONTRACT_TYPE_ID', _top(s, 'CONTRACT_TYPE_ID', 6), lambda v: SOZLESME_ISIM.get(v, f'Sözleşme kodu {v}'), series=s['CONTRACT_TYPE_ID']))
    D.append(kat_dim('chg', 'Ücretlendirme', 'CHARGING_METHOD_ID', _top(s, 'CHARGING_METHOD_ID', 6), lambda v: UCRET_ISIM.get(v, f'Ücretlendirme kodu {v}'), series=s['CHARGING_METHOD_ID']))
    D.append(kat_dim('dig', 'Dijital segment', 'DIGITAL_SEGMENT_NAME', _top(s, 'DIGITAL_SEGMENT_NAME', 8), str, series=s['DIGITAL_SEGMENT_NAME']))
    D.append(kat_dim('rah', 'Rahat segment', 'RAHAT_SEGMENT', _top(s, 'RAHAT_SEGMENT', 6), str, series=s['RAHAT_SEGMENT']))
    if 'CIHAZ_STD' in df and s['CIHAZ_STD'].notna().any():
        D.append(kat_dim('dev', 'Cihaz markası', 'CIHAZ_STD', _top(s, 'CIHAZ_STD', 8), str, series=s['CIHAZ_STD']))
    ten_ok = s['SUBSCRIBERS_LAST_TENURE_MONTHS'].notna().any()
    if ten_ok: D.append(dict(key='ten', name='Kıdem', labels=['0-6 ay', '6-12 ay', '1-2 yıl', '2+ yıl'], fn=lambda d: np.digitize(d['SUBSCRIBERS_LAST_TENURE_MONTHS'].fillna(0).to_numpy(), [6, 12, 24]).astype(np.uint8)))
    ea = _kenarlar(s['ARPU_TL'].dropna()); eg = _kenarlar(s['TOT_DATA_VOL_GB'].dropna())
    D.append(dict(key='ab', name='ARPU bandı', labels=_etiket_bant(ea, '₺'), fn=lambda d, e=ea: np.digitize(d['ARPU_TL'].fillna(0).to_numpy(), e).astype(np.uint8)))
    D.append(dict(key='gbb', name='Data kullanım bandı', labels=_etiket_bant(eg, ' GB'), fn=lambda d, e=eg: np.digitize(d['TOT_DATA_VOL_GB'].fillna(0).to_numpy(), e).astype(np.uint8)))
    D.append(dict(key='at', name='ARPU eğilimi (son 3 ay)', labels=['ARPU düşüşte (−%10 ve altı)', 'ARPU sabit', 'ARPU artışta (+%10 ve üstü)'], fn=lambda d: _arpu_trend(d)))
    D.append(dict(key='comp', name='Şikayet', labels=['Şikayeti yok', 'Şikayeti var'], fn=lambda d: (d['COMPLAINT_FL'].fillna(0).to_numpy() > 0).astype(np.uint8)))
    D.append(dict(key='mnp', name='MNP geçmişi', labels=['MNP geçmişi yok', 'MNP geçmişi var'], fn=lambda d: ((d['PRE_MNP'].fillna(0).to_numpy() > 0) | (d['PIN_COUNT_HIST'].fillna(0).to_numpy() > 0) | (d['POUT_COUNT_HIST'].fillna(0).to_numpy() > 0)).astype(np.uint8)))
    D.append(dict(key='po3', name='Son 3 ay port-out', labels=['Port-out yok', 'Port-out var'], fn=lambda d: (d['POUT_COUNT_L3M'].fillna(0).to_numpy() > 0).astype(np.uint8)))
    D.append(dict(key='pi12', name='Son 12 ay port-in', labels=['Port-in yok', 'Port-in var'], fn=lambda d: (d['PIN_COUNT_L12M'].fillna(0).to_numpy() > 0).astype(np.uint8)))
    if s['BONUS_NAME'].notna().any():
        tb = _top(s, 'BONUS_NAME', 5); mp = {v: i + 1 for i, v in enumerate(tb)}
        D.append(dict(key='bon', name='Bonus', labels=['Bonusu yok'] + [str(v) for v in tb] + ['Diğer bonus'], fn=lambda d, mp=mp, o=len(tb) + 1: d['BONUS_NAME'].map(mp).fillna(o).where(d['BONUS_NAME'].notna(), 0).to_numpy().astype(np.uint8)))
    D = [x for x in D if x['key'] not in KIRILIM_DISI]
    return D


def _arpu_trend(d):
    base = d[['ARPU_L1M', 'ARPU_L2M', 'ARPU_L3M']].mean(axis=1).to_numpy(); r = d['ARPU_TL'].to_numpy() / np.where(base > 0, base, np.nan)
    out = np.ones(len(d), np.uint8); out[r < .90] = 0; out[r > 1.10] = 2; return out


def hazirla(df, dims, srch):
    P = dict(n=len(df), id=df[ANAHTAR].to_numpy(), state=durum_kodu(df), giris=giris_kodu(df))
    P['dc'] = np.stack([d['fn'](df) for d in dims], 1).astype(np.uint8)
    f = lambda c: df[c].fillna(0).to_numpy(float)
    base = df[['ARPU_L1M', 'ARPU_L2M', 'ARPU_L3M']].mean(axis=1).to_numpy(); ratio = np.where(base > 0, df['ARPU_TL'].to_numpy() / np.where(base > 0, base, 1) - 1, 0)
    P['X'] = np.stack([f('ARPU_TL'), f('TOT_DATA_VOL_GB'), f('TOT_VOI_MOU'), f('TARIFF_OPTION_PRICE'), f('SUBSCRIBERS_LAST_TENURE_MONTHS'), f(YAS_KOLONU), np.clip(np.nan_to_num(ratio) * 100, -100, 300),
                       (f('COMPLAINT_FL') > 0).astype(float), (f('POUT_COUNT_L3M') > 0).astype(float), (f('PIN_COUNT_L12M') > 0).astype(float)], 1)
    P['S'] = df[srch].fillna(0).to_numpy(np.float32) if srch else np.zeros((len(df), 0), np.float32)
    P['arpu_l'] = df[['ARPU_L6M', 'ARPU_L5M', 'ARPU_L4M', 'ARPU_L3M', 'ARPU_L2M', 'ARPU_L1M', 'ARPU_TL']].to_numpy(np.float32)
    return P


def _bc(key, w, size):
    return np.bincount(key, weights=w, minlength=size)


def uret(dosyalar, oku_fn):
    NT = len(dosyalar); ay_ids = [m for m, _ in dosyalar]; bilgiler = []
    df0, b0 = oku_fn(dosyalar[-1][1], dosyalar[-1][0]); aktif_durum_belirle(df0); dims = dims_tanimla(df0)
    yas_ozet = df0[YAS_KOLONU].describe()[['min', '50%', 'max']].round(1).to_dict() if YAS_KOLONU in df0 and df0[YAS_KOLONU].notna().any() else None; ND = len(dims); K = {d['key']: i for i, d in enumerate(dims)}
    srch = sorted([c for c in df0.columns if c.endswith('_SEARCH')]); ilgi_lbl = [ILGI_ETIKET.get(c[:-7], c[:-7].title()) for c in srch]; NI = len(srch); del df0
    metr_ok = [True] * NMET
    ST = dict(t=[], d=[], c=[], n=[], s=[], q=[], i=[], u=[]); EVT = dict(t=[], e=[], d=[], c=[], n=[], s=[]); EVI = dict(t=[], e=[], n=[], s=[], i=[], u=[])
    QU = []; H = []; diag = []; prev = None; cross = None; arpu_hist = None
    for t, (mid, yol) in enumerate(dosyalar):
        df, bilgi = oku_fn(yol, mid)
        if t == 0:
            for j, (k, _, _) in enumerate(METR):
                col = dict(arpu='ARPU_TL', gb='TOT_DATA_VOL_GB', mou='TOT_VOI_MOU', price='TARIFF_OPTION_PRICE', ten='SUBSCRIBERS_LAST_TENURE_MONTHS', age=YAS_KOLONU, arpur='ARPU_L1M', comp='COMPLAINT_FL', po3='POUT_COUNT_L3M', pi12='PIN_COUNT_L12M')[k]
                metr_ok[j] = bool(df[col].notna().any())
        P = hazirla(df, dims, srch); st = P['state']; s0 = st == 0
        if not s0.any():
            raise ValueError(f"{mid}: portföyde (aktif durumlu) hiç satır yok. SUBSCRIBER_STATUS_ID dağılımı: {dict(df['SUBSCRIBER_STATUS_ID'].value_counts().head(8))}; AKTIF_DURUM_IDS={AKTIF_DURUM_IDS}")
        pst = None if prev is None else prev['id'][prev['state'] == 0]
        arr = np.zeros(P['n'], bool) if pst is None else (s0 & ~np.isin(P['id'], pst))
        evm = {}                                                          # olay maskeleri (t. satırları)
        if prev is not None:
            for code, nm in [(1, 'in_acq'), (2, 'in_portin'), (3, 'in_pre'), (4, 'in_post'), (5, 'in_oth')]: evm[nm] = arr & (P['giris'] == code)
            for code, nm in [(1, 'out_pre'), (2, 'out_post'), (3, 'out_rival'), (4, 'out_close')]: evm[nm] = (st == code)
            ghost = ~np.isin(pst, P['id']); gmask_prev = np.zeros(prev['n'], bool); gmask_prev[np.where(prev['state'] == 0)[0][ghost]] = True
        def grp(mask, src, with_stock):
            for d in range(ND + 1):
                key = np.zeros(int(mask.sum()), np.int64) if d == 0 else src['dc'][mask, d - 1].astype(np.int64); size = 1 if d == 0 else len(dims[d - 1]['labels'])
                n = _bc(key, None, size); X = src['X'][mask]; sm = np.stack([_bc(key, X[:, j], size) for j in range(NMET)], 1)
                yield d, size, n, sm, key
        # ── portföy (stock) ──
        for d, size, n, sm, key in grp(s0, P, True):
            X = P['X'][s0]; sq = np.stack([_bc(key, X[:, j] ** 2, size) for j in range(NMET)], 1); Ss = P['S'][s0]
            isum = np.stack([_bc(key, Ss[:, j], size) for j in range(NI)], 1) if NI else np.zeros((size, 0)); iuse = np.stack([_bc(key, (Ss[:, j] > 0).astype(float), size) for j in range(NI)], 1) if NI else np.zeros((size, 0))
            for c in range(size):
                if n[c] > 0:
                    ST['t'].append(t); ST['d'].append(d); ST['c'].append(c); ST['n'].append(int(n[c])); ST['s'].append(np.round(sm[c], 2).tolist()); ST['q'].append(np.round(sq[c], 1).tolist())
                    ST['i'].append(np.round(isum[c]).astype(int).tolist()); ST['u'].append(np.round(iuse[c]).astype(int).tolist())
        X = P['X'][s0]; QU.append({k: np.round(np.quantile(X[:, j], [.1, .25, .5, .75, .9]), 2).tolist() for j, (k, _, _) in enumerate(METR) if k in ('arpu', 'gb', 'mou', 'ten', 'age')})
        o = np.argsort(P['id'][s0]); H.append(dict(id=P['id'][s0][o], arpu=P['X'][s0][o, 0], **{k: P['dc'][s0][o, K[k]] for k in TRANS_DIMS if k in K}))
        # ── olaylar ──
        cnt = {}
        if prev is not None:
            for ei, nm in enumerate(EV):
                if nm == 'stock': continue
                if nm == 'out_ghost': mask, src = gmask_prev, prev
                else: mask, src = evm[nm], P
                cnt[nm] = int(mask.sum())
                if cnt[nm] == 0: continue
                for d, size, n, sm, key in grp(mask, src, False):
                    for c in range(size):
                        if n[c] > 0: EVT['t'].append(t); EVT['e'].append(ei); EVT['d'].append(d); EVT['c'].append(c); EVT['n'].append(int(n[c])); EVT['s'].append(np.round(sm[c], 2).tolist())
                Ss = src['S'][mask]
                EVI['t'].append(t); EVI['e'].append(ei); EVI['n'].append(cnt[nm]); EVI['s'].append(np.round(src['X'][mask].sum(0), 2).tolist())
                EVI['i'].append(np.round(Ss.sum(0)).astype(int).tolist() if NI else []); EVI['u'].append(np.round((Ss > 0).sum(0)).astype(int).tolist() if NI else [])
        diag.append(dict(ay=mid, satir=P['n'], portfoy=int(s0.sum()), dur_pre=int((st == 1).sum()), dur_post=int((st == 2).sum()), dur_rakip=int((st == 3).sum()), dur_pasif=int((st == 4).sum()), **{k: cnt.get(k, 0) for k in EV[1:]}, tekrar_eden_id=bilgi['tekrar_eden_id']))
        bilgiler.append(bilgi)
        if t == NT - 1:
            al = P['arpu_l'][s0]; arpu_hist = [dict(lag=l, ort=float(np.nanmean(al[:, i])) if np.isfinite(al[:, i]).any() else None, n=int(np.isfinite(al[:, i]).sum())) for i, l in enumerate(['L6M', 'L5M', 'L4M', 'L3M', 'L2M', 'L1M', 'Bu ay'])]
            cross = []
            for a, b in CROSS:
                if a in K and b in K:
                    na, nb = len(dims[K[a]]['labels']), len(dims[K[b]]['labels']); key = P['dc'][s0, K[a]].astype(np.int64) * nb + P['dc'][s0, K[b]]
                    cross.append(dict(a=K[a], b=K[b], n=_bc(key, None, na * nb).astype(int).tolist(), arpu=np.round(_bc(key, P['X'][s0, 0], na * nb), 1).tolist()))
        prev = dict(n=P['n'], id=P['id'], state=st, dc=P['dc'], X=P['X'], S=P['S'])
    # ── ortak müşteri geçişleri (tüm ay çiftleri) ──
    pairs = []; ahist = []
    for i in range(NT):
        for j in range(i + 1, NT):
            _, ia, ib = np.intersect1d(H[i]['id'], H[j]['id'], assume_unique=True, return_indices=True)
            if len(ia) == 0: continue
            r = H[j]['arpu'][ib] / np.where(H[i]['arpu'][ia] > 0, H[i]['arpu'][ia], np.nan) - 1; r = r[np.isfinite(r)]
            ahist.append(dict(a=i, b=j, n=int(len(ia)), h=np.histogram(np.clip(r, -1, 1e8), bins=HIST_EDGES)[0].astype(int).tolist(), up=float((r > .05).mean()), down=float((r < -.05).mean()), med=float(np.median(r)), mean=float(r.mean())))
            for k in TRANS_DIMS:
                if k in K:
                    nl = len(dims[K[k]]['labels']); m = np.bincount(H[i][k][ia].astype(np.int64) * nl + H[j][k][ib], minlength=nl * nl)
                    pairs.append(dict(a=i, b=j, d=K[k], m=m.astype(int).tolist()))
    return dict(yas_ozet=yas_ozet, dims=dims, ay_ids=ay_ids, ilgi=ilgi_lbl, ST=ST, EVT=EVT, EVI=EVI, QU=QU, cross=cross, arpu_hist=arpu_hist, pairs=pairs, ahist=ahist, metr_ok=metr_ok,
                diag=pd.DataFrame(diag), bilgi=pd.DataFrame(bilgiler))

### 4 · Tanılama (kişisel veri içermez)
Hata alırsanız bu özet otomatik basılır; paylaşılabilir.

In [ ]:
# ───────────── TANILAMA (kişisel veri içermez) ─────────────
def tanilama(dosyalar, nrows=20000):
    """Dosya yapısı özeti: sürümler, kolonlar, tipler, boşluk oranları ve küçük kategorik kolonların değer sayıları. Müşteri satırı / ID / telefon içermez."""
    import sys
    print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")
    print(f"Dosyalar: {[os.path.basename(p) for _, p in dosyalar]}")
    kucuk = ['SUBSCRIBER_STATUS_ID', 'CONTRACT_TYPE_ID', 'CHARGING_METHOD_ID', 'COMPLAINT_FL', 'PRE_MNP'] + [f'{b}_FL' for b in BAYRAKLAR] + CINSIYET_ADAYLARI
    for mid, yol in [dosyalar[0], dosyalar[-1]]:
        try:
            enc = _kodlama(yol); sep = _ayrac(yol, enc)
            ham = pd.read_csv(yol, sep=sep, encoding=enc, nrows=nrows, low_memory=False); ham.columns = [kolon_norm(c) for c in ham.columns]
        except Exception as e:
            print(f"\n[{mid}] okunamadı: {type(e).__name__}: {e}"); continue
        print(f"\n[{mid}] kodlama={enc} ayraç={sep!r} · ilk {len(ham)} satır · {ham.shape[1]} kolon")
        eksik = [c for c in BEKLENEN if c not in ham.columns and c != 'PHONE']
        print("  Beklenip bulunamayan kolonlar:", eksik if eksik else "yok")
        print("  Beklenmeyen ek kolonlar:", [c for c in ham.columns if c not in BEKLENEN and not c.endswith('_SEARCH')][:40])
        if YAS_KOLONU in ham.columns: print(f"  {YAS_KOLONU} (min/medyan/maks): {ham[YAS_KOLONU].min()} / {ham[YAS_KOLONU].median()} / {ham[YAS_KOLONU].max()}")
        gizli = {'PHONE', ANAHTAR, KAMPANYA_KOLON, 'SUBSCRIBER_ID', 'CAMPAIGN_SUBSCRIBER_ID', 'MUSTERI_ANAHTAR'}
        tip = pd.DataFrame({'tip': ham.dtypes.astype(str), 'boş_%': (ham.isna().mean() * 100).round(1)}).loc[[c for c in ham.columns if c not in gizli]]
        if (mid, yol) == dosyalar[-1]:
            print("  Kolon tipleri ve boşluk oranı (yalnız son ay):"); print(tip.to_string())
        sade = [c for c in kucuk + ['CITY', 'DIGITAL_SEGMENT_NAME', 'RAHAT_SEGMENT'] + CIHAZ_ADAYLARI + BONUS_ADAYLARI if c in ham.columns]
        try:
            tam = pd.read_csv(yol, sep=sep, encoding=enc, usecols=lambda c: kolon_norm(c) in set(sade), low_memory=False); tam.columns = [kolon_norm(c) for c in tam.columns]
        except Exception:
            tam = ham
        print(f"  ── Tüm dosya üzerinden sayımlar ({len(tam):,} satır) ──")
        for c in kucuk:
            if c in tam.columns:
                print(f"  {c} değerleri:", tam[c].value_counts(dropna=False).head(8).to_dict())
        for c in ['CITY', 'DIGITAL_SEGMENT_NAME', 'RAHAT_SEGMENT'] + CIHAZ_ADAYLARI + BONUS_ADAYLARI:
            if c in tam.columns:
                print(f"  {c}: {tam[c].nunique()} farklı değer, en sık: {list(tam[c].value_counts().head(5).index)}")

### 5 · Çalıştır ve veri doğrulama

In [ ]:
# ───────────── ÇALIŞTIR: CSV'ler → rapor tabloları ─────────────
DOSYALAR = dosyalari_bul(VERI_KLASORU)
anahtar_sec(DOSYALAR)
print("Bulunan dosyalar:", [os.path.basename(p) for _, p in DOSYALAR])
if TANILAMA:
    tanilama(DOSYALAR)
try:
    R = uret(DOSYALAR, csv_oku)
except Exception:
    import traceback
    traceback.print_exc()
    print("\n── Tanılama özeti (kişisel veri içermez) ──")
    tanilama(DOSYALAR)
    raise
AY_IDS = R['ay_ids']
# ── veri doğrulama raporu ──
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
print("\n── Dosya okuma ──"); print(R['bilgi'][['ay', 'dosya', 'kodlama', 'ayrac', 'ondalik', 'satir', 'kolon', 'tekrar_eden_id', 'arama_kolonu']].to_string(index=False))
_eks = sorted({c for l in R['bilgi']['eksik_kolonlar'] for c in l})
print("\nEksik kolonlar:", _eks if _eks else "yok")
print("Cinsiyet kolonu:", R['bilgi']['cinsiyet_kolonu'].iloc[-1], "| Cihaz markası kolonu:", R['bilgi']['cihaz_kolonu'].iloc[-1])
print("Tarife adı kolonu:", R['bilgi']['tarife_adi_kolonu'].iloc[-1])
print(f"Yaş kolonu {YAS_KOLONU} (min / medyan / maks): {R['yas_ozet']}  ← kişi yaşı (yıl) olmalı; ay cinsinden abonelik süresiyse YAS_KOLONU'nu değiştirin")
print("Kırılımlar:", [d['name'] for d in R['dims']])
print("\n── Hareket özeti (ay bazında müşteri adedi) ──"); print(R['diag'].to_string(index=False))

# ── müşteri adedi uzlaşması: ham satırdan portföye adım adım ──
_u = pd.concat([R['bilgi'][['ay', 'ham_satir', 'benzersiz_anahtar', 'benzersiz_kampanya', 'tekrar_eden_id']].reset_index(drop=True),
                R['diag'][['portfoy', 'dur_pre', 'dur_post', 'dur_rakip', 'dur_pasif']].reset_index(drop=True)], axis=1)
_u.columns = ['ay', 'ham_satır', 'benzersiz_ANAHTAR', f'benzersiz_{KAMPANYA_KOLON}', 'tekrar_atılan', 'PORTFÖY', "Prepaid'e(bayrak)", "Postpaid'e(bayrak)", 'rakip(pasif+port-out)', 'pasif/iptal']
print("\n── Müşteri adedi uzlaşması (ham satır → portföy) ──"); print(_u.to_string(index=False))
_son = _u.iloc[-1]
_disi = int(_son["Prepaid'e(bayrak)"] + _son["Postpaid'e(bayrak)"] + _son['rakip(pasif+port-out)'] + _son['pasif/iptal'])
print(f"\nSon ay: {int(_son['ham_satır']):,} satırdan {int(_son['tekrar_atılan']):,} tekrar atıldı; {int(_son['PORTFÖY']):,} satır portföyde kaldı (portföy dışı sayılan: {_disi:,}).")
if _son['tekrar_atılan'] > .005 * _son['ham_satır']:
    print(f"⚠ {ANAHTAR_KAYNAK} aynı müşteri için birden fazla satırda tekrar ediyor (benzersiz {int(_son['benzersiz_ANAHTAR']):,}, {KAMPANYA_KOLON} benzersiz {_son[f'benzersiz_{KAMPANYA_KOLON}']}). "
          f"Her satır ayrı müşteri/hat ise ANAHTAR_KOLONU ayarını benzersiz kolonla değiştirin; toplamlar tekrar atılan satırlar kadar eksik çıkıyor olabilir.")
if _disi / _son['ham_satır'] > .05:
    print(f"⚠ Satırların %{100*_disi/_son['ham_satır']:.0f}'i portföy dışı sayıldı (aktif olmayan durum veya çıkış bayrağı). AKTIF_DURUM_IDS ve çıkış bayraklarının anlamını kontrol edin.")

### 6 · Rapor verisi

In [ ]:
# ───────────── RAPOR VERİSİ ─────────────
def _blok(obj):
    ham = json.dumps(obj, ensure_ascii=False, separators=(',', ':')).encode('utf-8')
    if VERI_SIKISTIR:
        return base64.b64encode(gzip.compress(ham, 9)).decode('ascii')
    return ham.decode('utf-8').replace('</', '<\\/')

_TR = ['Oca', 'Şub', 'Mar', 'Nis', 'May', 'Haz', 'Tem', 'Ağu', 'Eyl', 'Eki', 'Kas', 'Ara']
meta = dict(segment=SEGMENT_ADI, uretim=dt.datetime.now().strftime('%d.%m.%Y %H:%M'), sentetik=False,
    months=[dict(id=str(m), l=f"{_TR[int(str(m)[4:6])-1]} {str(m)[:4]}", s=f"{_TR[int(str(m)[4:6])-1]} {str(m)[2:4]}") for m in AY_IDS],
    dims=[dict(key=d['key'], name=d['name'], labels=d['labels']) for d in R['dims']], metrics=[dict(key=k, name=n, unit=u, ok=o) for (k, n, u), o in zip(METR, R['metr_ok'])],
    events=EV, eventLabels=EV_LBL, ilgi=R['ilgi'], alert=dict(adet=UYARI_ADET_ESIK_PCT, ort=UYARI_ORT_ESIK_PCT, min=UYARI_MIN_GRUP), histEdges=[-1.0, -.5, -.3, -.2, -.1, -.05, .05, .1, .2, .3, .5, 1.0, 9.99])
veri1 = dict(meta=meta, stock=R['ST'], quant=R['QU'], cross=R['cross'], arpuHist=R['arpu_hist'], pairs=R['pairs'], ahist=R['ahist'])
veri2 = dict(ev=R['EVT'], evi=R['EVI'])
PAYLOAD = [_blok(veri1), _blok(veri2)]
print("Gömülen veri boyutu:", [f"{len(p)/1e6:.2f} MB" for p in PAYLOAD])

### 7 · Arayüz şablonu (HTML/CSS/JS)

In [ ]:
HTML_SABLON = r'''<!doctype html>
<html lang="tr">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>__TITLE__</title>
<style>
:root{--navy:#0A2A5E;--blue:#0057B8;--sky:#4DA3FF;--yellow:#FFC72C;--yellow-l:#FFF4CC;--bg:#F3F6FB;--card:#FFFFFF;--line:#E2E8F3;--text:#0F2347;--mut:#5E6E8C;--good:#13A97B;--bad:#E5484D;
  --font:"Segoe UI","Helvetica Neue",Helvetica,Arial,system-ui,sans-serif;--shadow:0 1px 2px rgba(10,42,94,.05),0 6px 20px rgba(10,42,94,.05)}
*{box-sizing:border-box}
html,body{margin:0}
body{background:var(--bg);color:var(--text);font:14px/1.5 var(--font);-webkit-font-smoothing:antialiased}
.app{display:grid;grid-template-columns:252px 1fr;min-height:100vh}
/* ── sol menü ── */
.nav{background:#fff;border-right:1px solid var(--line);padding:22px 14px;position:sticky;top:0;height:100vh;display:flex;flex-direction:column;gap:6px;overflow:auto}
.brand{display:flex;gap:12px;align-items:center;padding:0 8px 18px;border-bottom:1px solid var(--line);margin-bottom:12px}
.mark{width:40px;height:40px;border-radius:12px;background:var(--navy);display:grid;place-items:center;position:relative;flex:none}
.mark::after{content:"";position:absolute;right:-4px;bottom:-4px;width:16px;height:16px;border-radius:5px;background:var(--yellow);border:2px solid #fff}
.mark svg{width:22px;height:22px}
.brand b{display:block;font-size:15px;color:var(--navy);line-height:1.2}
.brand span{font-size:11.5px;color:var(--mut)}
.nav .sec{font-size:10.5px;letter-spacing:.14em;text-transform:uppercase;color:var(--mut);padding:10px 10px 4px}
.nb{display:flex;gap:12px;align-items:flex-start;padding:11px 12px;border-radius:12px;cursor:pointer;border:0;background:none;text-align:left;font:inherit;color:var(--text);position:relative;width:100%}
.nb:hover{background:#F3F6FB}
.nb .no{flex:none;width:26px;height:26px;border-radius:8px;background:#EAF0FA;color:var(--blue);display:grid;place-items:center;font-weight:700;font-size:12.5px;margin-top:1px}
.nb b{display:block;font-size:14px;font-weight:650}
.nb small{display:block;font-size:11.8px;color:var(--mut);line-height:1.35;margin-top:1px}
.nb.on{background:var(--yellow-l)}
.nb.on::before{content:"";position:absolute;left:-14px;top:10px;bottom:10px;width:4px;border-radius:0 4px 4px 0;background:var(--yellow)}
.nb.on .no{background:var(--blue);color:#fff}
.nav .foot{margin-top:auto;padding:14px 10px 0;font-size:11.5px;color:var(--mut);border-top:1px solid var(--line)}
.tag{display:inline-block;background:#EAF0FA;color:var(--blue);padding:2px 9px;border-radius:99px;font-size:11px;font-weight:600}
.tag.warn{background:#FFF0D6;color:#9A5B00}
/* ── ana alan ── */
.main{min-width:0}
.head{background:#fff;border-bottom:1px solid var(--line);border-top:4px solid var(--yellow);padding:18px 30px 0}
.head h1{margin:0;font-size:22px;color:var(--navy);letter-spacing:-.01em}
.head p{margin:3px 0 0;color:var(--mut);font-size:13px}
.ctl{display:flex;gap:18px;flex-wrap:wrap;align-items:flex-end;padding:16px 0 14px}
.fld{display:flex;flex-direction:column;gap:4px}
.fld>label{font-size:10.5px;letter-spacing:.1em;text-transform:uppercase;color:var(--mut);font-weight:650}
select,.btn{appearance:none;-webkit-appearance:none;border:1px solid #CBD6EA;background:#fff;color:var(--text);border-radius:10px;padding:8px 30px 8px 12px;font:inherit;font-weight:600;cursor:pointer;min-width:112px;
  background-image:url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='10' height='6' fill='none' stroke='%235E6E8C' stroke-width='1.8'%3E%3Cpath d='M1 1l4 4 4-4'/%3E%3C/svg%3E");background-repeat:no-repeat;background-position:right 11px center}
select:hover,.btn:hover{border-color:var(--blue)}
.btn{background-image:none;padding:8px 14px;min-width:0}
.btn.pri{background:var(--blue);border-color:var(--blue);color:#fff}
.ms{position:relative}
.ms-b{display:flex;align-items:center;gap:6px;border:1px solid #CBD6EA;background:#fff;border-radius:10px;padding:8px 12px;font:inherit;cursor:pointer;font-weight:600;color:var(--text)}
.ms-b span{color:var(--mut);font-weight:500}
.ms-b.on{border-color:var(--blue);background:#EAF2FF}.ms-b.on span{color:var(--blue);font-weight:700}
.ms-p{display:none;position:absolute;top:calc(100% + 6px);left:0;min-width:200px;background:#fff;border:1px solid var(--line);border-radius:12px;padding:8px;box-shadow:0 12px 30px rgba(10,42,94,.15);z-index:60}
.ms.open .ms-p{display:block}
.ms-p label{display:flex;gap:9px;align-items:center;padding:6px 8px;border-radius:8px;cursor:pointer;font-size:13px}
.ms-p label:hover{background:var(--bg)}
.ms-p input{accent-color:var(--blue);width:15px;height:15px}
.ms-p .act{display:flex;justify-content:space-between;padding:2px 8px 7px;border-bottom:1px solid var(--line);margin-bottom:4px;font-size:12px}
.ms-p .act a{color:var(--blue);font-weight:600;cursor:pointer}
.scope{padding:0 0 12px;color:var(--mut);font-size:12.8px}
.scope b{color:var(--text)}.scope .fx{color:var(--blue);font-weight:600}
.sub{display:flex;gap:2px;margin-top:2px;overflow-x:auto}
.sub button{background:none;border:0;padding:11px 18px;font:inherit;font-weight:600;color:var(--mut);cursor:pointer;border-bottom:3px solid transparent;white-space:nowrap}
.sub button:hover{color:var(--text)}
.sub button.on{color:var(--blue);border-bottom-color:var(--blue)}
.body{padding:24px 30px 60px}
.page{display:none}.page.on{display:block}
.tab{display:none}.tab.on{display:block;animation:f .25s ease}
@keyframes f{from{opacity:0;transform:translateY(5px)}to{opacity:1;transform:none}}
/* ── kartlar ── */
.grid{display:grid;grid-template-columns:repeat(12,1fr);gap:18px}
.c3{grid-column:span 3}.c4{grid-column:span 4}.c5{grid-column:span 5}.c6{grid-column:span 6}.c7{grid-column:span 7}.c8{grid-column:span 8}.c12{grid-column:span 12}
.card{background:var(--card);border:1px solid var(--line);border-radius:16px;padding:18px 18px 12px;box-shadow:var(--shadow);min-width:0}
.card h3{margin:0;font-size:15px;color:var(--navy);font-weight:700}
.card .d{margin:2px 0 8px;font-size:12.5px;color:var(--mut)}
.ch{width:100%;height:320px}.ch.t{height:400px}.ch.s{height:260px}.ch.k{height:500px}
.tools{display:flex;gap:10px;align-items:center;margin-bottom:10px;flex-wrap:wrap}
.seg{display:inline-flex;background:#EAF0FA;border-radius:10px;padding:3px}
.seg button{border:0;background:none;padding:6px 12px;border-radius:8px;font:inherit;font-size:12.5px;font-weight:600;color:var(--mut);cursor:pointer}
.seg button.on{background:#fff;color:var(--blue);box-shadow:0 1px 3px rgba(10,42,94,.15)}
/* KPI */
.kpis{display:grid;grid-template-columns:repeat(6,1fr);gap:14px;margin-bottom:18px}
.kpi{background:#fff;border:1px solid var(--line);border-radius:16px;padding:14px 14px 0;box-shadow:var(--shadow);overflow:hidden;position:relative}
.kpi .l{font-size:11px;letter-spacing:.06em;text-transform:uppercase;color:var(--mut);font-weight:650}
.kpi .v{font-size:25px;font-weight:750;color:var(--navy);margin:5px 0 6px;letter-spacing:-.02em;font-variant-numeric:tabular-nums}
.kpi .dl{display:flex;gap:7px;align-items:center;font-size:11.5px;color:var(--mut)}
.pill{padding:1px 8px;border-radius:99px;font-weight:700;font-size:11.5px}
.pill.g{background:#DDF5EC;color:#0B7F5B}.pill.b{background:#FDE4E5;color:#B4262B}.pill.n{background:#EDF1F8;color:var(--mut)}
.kpi svg{display:block;width:calc(100% + 28px);height:34px;margin:8px -14px 0}
.kpi.hi{border-top:3px solid var(--yellow)}
/* tablolar */
table{width:100%;border-collapse:collapse;font-size:13px;font-variant-numeric:tabular-nums}
th{font-size:10.8px;letter-spacing:.06em;text-transform:uppercase;color:var(--mut);font-weight:650;text-align:right;padding:9px 9px;border-bottom:1px solid var(--line);cursor:pointer;white-space:nowrap;background:#FAFBFE}
th.sorted::after{content:" ↓";color:var(--blue)}
td{text-align:right;padding:9px 9px;border-bottom:1px solid #EEF2F9}
th:first-child,td:first-child{text-align:left}
tr:hover td{background:#F7FAFF}
.up{color:var(--good);font-weight:600}.dn{color:var(--bad);font-weight:600}.z{color:#B5C0D6}
.ins{display:flex;gap:12px;padding:11px 2px;border-bottom:1px solid var(--line)}
.ins:last-child{border:0}
.ins .i{flex:none;width:30px;height:30px;border-radius:9px;display:grid;place-items:center;font-weight:800;font-size:14px;background:#EAF0FA;color:var(--blue)}
.ins.g .i{background:#DDF5EC;color:var(--good)}.ins.b .i{background:#FDE4E5;color:var(--bad)}
.ins b{color:var(--navy)}.ins div div{color:var(--mut);font-size:12.8px}
.note{background:#FFF9E3;border:1px solid #F6E3A0;color:#6B5200;padding:9px 13px;border-radius:10px;font-size:12.8px;margin-bottom:16px}
.load{position:fixed;inset:0;background:var(--bg);display:grid;place-items:center;z-index:99;color:var(--mut);font-size:14px}
.load b{display:block;color:var(--navy);font-size:18px;margin-bottom:6px;text-align:center}

.al{display:flex;gap:12px;padding:12px 4px;border-bottom:1px solid var(--line)}
.al:last-child{border:0}
.al .i{flex:none;width:32px;height:32px;border-radius:10px;display:grid;place-items:center;font-weight:800;font-size:15px}
.al.up .i{background:#DDF5EC;color:var(--good)}.al.dn .i{background:#FDE4E5;color:var(--bad)}.al.nt .i{background:#EAF0FA;color:var(--blue)}
.al b{color:var(--navy)}.al .tg{display:inline-block;background:#EAF0FA;color:var(--blue);border-radius:99px;font-size:11px;font-weight:700;padding:1px 9px;margin-right:6px}
.al .sm{color:var(--mut);font-size:12.5px;margin-top:2px}
.sl{display:flex;flex-direction:column;gap:4px;min-width:190px}
.sl input{accent-color:var(--blue);width:100%}
.sl .v{font-weight:700;color:var(--blue)}
.h2{font-size:17px;font-weight:700;color:var(--navy);margin:30px 0 4px}
.chk{display:flex;gap:14px;align-items:center;font-size:13px}
.chk label{display:flex;gap:6px;align-items:center;cursor:pointer}
.chk input{accent-color:var(--blue)}
td.hi{background:rgba(229,72,77,.10)}td.lo{background:rgba(0,87,184,.09)}
@media(max-width:1180px){.kpis{grid-template-columns:repeat(3,1fr)}}
@media(max-width:960px){.app{grid-template-columns:1fr}.nav{position:static;height:auto;flex-direction:row;flex-wrap:wrap}.nav .foot,.nav .sec{display:none}.c3,.c4,.c5,.c6,.c7,.c8{grid-column:span 12}.kpis{grid-template-columns:repeat(2,1fr)}.body,.head{padding-left:16px;padding-right:16px}}
@media print{.nav,.ctl,.sub,.tools{display:none}.app{display:block}.page{display:block!important}.tab{display:block!important}}
</style>
__PLOTLY__
</head>
<body>
<div class="load" id="load"><div><b>Veriler yükleniyor…</b>Rapor hazırlanıyor</div></div>
<div class="app">
  <aside class="nav">
    <div class="brand"><div class="mark"><svg viewBox="0 0 24 24" fill="none" stroke="#fff" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round"><path d="M3 17l5-6 4 3 8-9"/><path d="M15 5h5v5"/></svg></div><div><b>Segment Monitörü</b><span id="segName"></span></div></div>
    <div class="sec">Analizler</div>
    <button class="nb on" data-p="p1"><span class="no">1</span><span><b>Ana Ekran</b><small id="nb1">Son ayın güncel durumu</small></span></button>
    <button class="nb" data-p="p2"><span class="no">2</span><span><b>Karşılaştırma</b><small>İki ayı karşılaştır, değişimleri ve uyarıları gör</small></span></button>
    <button class="nb" data-p="p3"><span class="no">3</span><span><b>Müşteri Hareketleri</b><small>Şirket içi / dışı gelen ve gidenlerin profili</small></span></button>
    <div class="foot"><span class="tag warn" id="synTag">Sentetik veri</span><div style="margin-top:8px" id="genAt"></div></div>
  </aside>
  <div class="main">

  <section class="page on" id="p1">
    <div class="head"><h1>Ana Ekran</h1><p id="p1sub"></p>
      <div class="sub" id="p1tabs"><button data-t="g" class="on">Genel görünüm</button><button data-t="p">Müşteri profili</button><button data-t="a">ARPU ve kullanım</button><button data-t="t">Dijital segment, cihaz ve Rahat</button><button data-t="i">Dijital ilgi alanları</button><button data-t="r">Risk sinyalleri ve karne</button></div></div>
    <div class="body">
      <div class="tab on" id="p1-g"><div class="kpis" id="g_kpis"></div><div class="grid" id="g_grid"></div></div>
      <div class="tab" id="p1-p"><div class="grid" id="p_grid"></div></div>
      <div class="tab" id="p1-a"><div class="ctl" style="padding-top:0"><div class="fld"><label>Kırılım</label><select id="a_dim"></select></div></div><div class="grid" id="a_grid"></div></div>
      <div class="tab" id="p1-t"><div class="grid" id="t_grid"></div></div>
      <div class="tab" id="p1-i"><div class="ctl" style="padding-top:0"><div class="fld"><label>Ölçü</label><div class="seg" id="i_met"><button data-v="avg" class="on">Müşteri başı arama adedi</button><button data-v="pen">Arama yapan müşteri payı (%)</button></div></div><div class="fld"><label>Isı haritası kırılımı</label><select id="i_dim"></select></div></div><div class="grid" id="i_grid"></div></div>
      <div class="tab" id="p1-r"><div class="ctl" style="padding-top:0"><div class="fld"><label>Kırılım</label><select id="r_dim"></select></div></div><div class="grid" id="r_grid"></div></div>
    </div>
  </section>

  <section class="page" id="p2">
    <div class="head"><h1>Karşılaştırma</h1><p>İki ayı yan yana koyun; adet, ortalama ve dağılım değişimlerini görün. En altta ciddi değişimler otomatik raporlanır.</p>
      <div class="ctl">
        <div class="fld"><label>Ay A (baz)</label><select id="p2a"></select></div><div class="fld"><label>Ay B (karşılaştırılan)</label><select id="p2b"></select></div>
        <div class="fld"><label>Kırılım</label><select id="p2d"></select></div>
      </div>
      <div class="scope" id="p2s"></div>
      <div class="sub" id="p2tabs"><button data-t="g" class="on">Genel fark</button><button data-t="d">Dağılım değişimi</button><button data-t="a">Ortalama değişimi</button><button data-t="o">Ortak müşteriler ve geçişler</button><button data-t="i">İlgi alanı değişimi</button></div></div>
    <div class="body">
      <div class="tab on" id="p2-g"><div class="kpis" id="c_kpis"></div><div class="grid" id="c_grid"></div></div>
      <div class="tab" id="p2-d"><div class="grid" id="d_grid"></div></div>
      <div class="tab" id="p2-a"><div class="ctl" style="padding-top:0"><div class="fld"><label>Ölçü (grup ortalamaları grafiği)</label><select id="m_met"></select></div></div><div class="grid" id="m_grid"></div></div>
      <div class="tab" id="p2-o"><div class="ctl" style="padding-top:0"><div class="fld"><label>Geçiş matrisi kırılımı</label><select id="o_dim"></select></div></div><div class="grid" id="o_grid"></div></div>
      <div class="tab" id="p2-i"><div class="grid" id="n_grid"></div></div>
      <div class="h2">Bilgilendirme · ciddi değişimler</div>
      <div class="card"><div class="d">A ve B aylarının adet ve ortalamaları tüm kırılımlarda taranır; eşiği aşan değişimler aşağıda listelenir. Eşikleri değiştirebilirsiniz.</div>
        <div class="ctl" style="padding:6px 0 10px">
          <div class="sl"><label style="font-size:10.5px;letter-spacing:.1em;text-transform:uppercase;color:var(--mut);font-weight:650">Adet değişimi eşiği: <span class="v" id="al_a_v"></span></label><input type="range" id="al_a" min="1" max="15" step="0.5"></div>
          <div class="sl"><label style="font-size:10.5px;letter-spacing:.1em;text-transform:uppercase;color:var(--mut);font-weight:650">Ortalama değişimi eşiği: <span class="v" id="al_o_v"></span></label><input type="range" id="al_o" min="1" max="25" step="0.5"></div>
          <div class="sl"><label style="font-size:10.5px;letter-spacing:.1em;text-transform:uppercase;color:var(--mut);font-weight:650">En az grup büyüklüğü: <span class="v" id="al_m_v"></span></label><input type="range" id="al_m" min="50" max="3000" step="50"></div>
          <div class="chk"><label><input type="checkbox" id="al_c" checked>Adet</label><label><input type="checkbox" id="al_mean" checked>Ortalama</label><label><input type="checkbox" id="al_i" checked>İlgi alanı</label></div>
        </div><div id="alerts"></div></div>
    </div>
  </section>

  <section class="page" id="p3">
    <div class="head"><h1>Müşteri Hareketleri</h1><p>Şirket içi ve şirket dışı gelen / giden müşteriler ve profilleri. Hareket kuralları notebook'taki doğrulama tablosuyla birebir aynıdır.</p>
      <div class="ctl">
        <div class="fld"><label>Dönem</label><select id="p3p"></select></div>
        <div class="fld"><label>Kırılım</label><select id="p3d"></select></div>
        <div class="fld"><label>Hareket grubu</label><select id="p3e"></select></div>
      </div>
      <div class="scope" id="p3s"></div>
      <div class="sub" id="p3tabs"><button data-t="g" class="on">Genel akış</button><button data-t="p">Hareket profili</button><button data-t="k">Kırılımlarda hareket</button><button data-t="i">İlgi alanları</button></div></div>
    <div class="body">
      <div class="tab on" id="p3-g"><div class="kpis" id="h_kpis"></div><div class="grid" id="h_grid"></div></div>
      <div class="tab" id="p3-p"><div class="ctl" style="padding-top:0"><div class="fld"><label>Ölçü (grafik)</label><select id="h_met"></select></div></div><div class="grid" id="hp_grid"></div></div>
      <div class="tab" id="p3-k"><div class="grid" id="hk_grid"></div></div>
      <div class="tab" id="p3-i"><div class="grid" id="hi_grid"></div></div>
    </div>
  </section>
  </div>
</div>

<script id="d1" type="application/octet-stream" data-enc="__ENC__">__DATA1__</script>
<script id="d2" type="application/octet-stream" data-enc="__ENC__">__DATA2__</script>
<script>
(async function(){
'use strict';
const $=id=>document.getElementById(id);
async function load(id){const n=$(id);let t=n.textContent.trim();
  if(n.dataset.enc==='gz'){const b=atob(t),u=new Uint8Array(b.length);for(let i=0;i<b.length;i++)u[i]=b.charCodeAt(i);
    t=await new Response(new Blob([u]).stream().pipeThrough(new DecompressionStream('gzip'))).text();}
  return JSON.parse(t);}
let D1,D2;
try{[D1,D2]=await Promise.all([load('d1'),load('d2')]);}
catch(e){$('load').innerHTML='<div><b>Veri açılamadı</b>Bu tarayıcı sıkıştırılmış veriyi desteklemiyor olabilir. Notebook ayarlarında <code>VERI_SIKISTIR = False</code> yapıp raporu yeniden üretin.<br><small>'+e+'</small></div>';return;}
const META=D1.meta,MONTHS=META.months,NT=MONTHS.length;
/* ── ortak sabitler ── */
const C={navy:'#0A2A5E',blue:'#0057B8',sky:'#4DA3FF',yellow:'#FFC72C',text:'#0F2347',mut:'#5E6E8C',grid:'#E8EDF6',cmp:'#B4C2DC',good:'#13A97B',bad:'#E5484D'};
const PAL=['#0057B8','#FFC72C','#4DA3FF','#0A2A5E','#13A97B','#FF8A3D','#8E7CC3','#8C97A8'];
const ST5=['#0057B8','#7B6CC4','#A99BDD','#E5484D','#8C97A8'];
const FONT='"Segoe UI","Helvetica Neue",Helvetica,Arial,sans-serif';
const nf0=new Intl.NumberFormat('tr-TR',{maximumFractionDigits:0}),nf1=new Intl.NumberFormat('tr-TR',{minimumFractionDigits:1,maximumFractionDigits:1}),nf2=new Intl.NumberFormat('tr-TR',{minimumFractionDigits:2,maximumFractionDigits:2});
const ok=v=>typeof v==='number'&&isFinite(v);
const fInt=v=>ok(v)?nf0.format(v):'–',fP1=v=>ok(v)?'%'+nf1.format(v):'–',fP2=v=>ok(v)?'%'+nf2.format(v):'–',fTL=v=>ok(v)?'₺'+nf0.format(v):'–';
const fS=v=>ok(v)?(v>0?'+':v<0?'−':'')+nf0.format(Math.abs(v)):'–',fPP=v=>ok(v)?(v>=.05?'+':v<=-.05?'−':'')+nf1.format(Math.abs(v))+' puan':'–';
const ML=i=>MONTHS[i].l,MS=i=>MONTHS[i].s,pct=(a,b)=>b>0?100*a/b:NaN,sum=a=>{let s=0;for(const x of a)s+=x;return s;};
const rgba=(h,a)=>{const n=parseInt(h.slice(1),16);return `rgba(${n>>16},${n>>8&255},${n&255},${a})`;};
/* ── Plotly yardımcıları ── */
const CFG={displayModeBar:false,responsive:true};
function merge(a,b){for(const k in b){if(b[k]&&typeof b[k]==='object'&&!Array.isArray(b[k]))a[k]=merge(a[k]&&typeof a[k]==='object'?a[k]:{},b[k]);else a[k]=b[k];}return a;}
function LAY(o){return merge({paper_bgcolor:'rgba(0,0,0,0)',plot_bgcolor:'rgba(0,0,0,0)',font:{family:FONT,color:C.text,size:12},separators:',.',margin:{l:54,r:14,t:8,b:40},colorway:PAL,
  xaxis:{gridcolor:'rgba(0,0,0,0)',linecolor:C.grid,tickfont:{color:C.mut},automargin:true,zeroline:false},
  yaxis:{gridcolor:C.grid,zeroline:false,tickfont:{color:C.mut},automargin:true},showlegend:true,
  legend:{orientation:'h',y:1.13,x:0,font:{size:11.5,color:C.mut},bgcolor:'rgba(0,0,0,0)'},hoverlabel:{bgcolor:C.navy,bordercolor:C.navy,font:{color:'#fff',family:FONT}}},o||{});}
function plot(id,tr,lay){const e=$(id);if(e)Plotly.react(e,tr,LAY(lay),CFG);}
const cc=i=>PAL[i%PAL.length];
function sparkSvg(vals){const v=vals.filter(ok);if(v.length<2)return '';const mn=Math.min(...v),mx=Math.max(...v),w=140,h=34,p=3,rg=(mx-mn)||1;
  const pts=vals.map((y,i)=>[p+i*(w-2*p)/(vals.length-1),h-p-(ok(y)?(y-mn)/rg:0)*(h-2*p-4)]);const d=pts.map((q,i)=>(i?'L':'M')+q[0].toFixed(1)+' '+q[1].toFixed(1)).join(' '),id='g'+Math.random().toString(36).slice(2,7),e=pts[pts.length-1];
  return `<svg viewBox="0 0 ${w} ${h}" preserveAspectRatio="none"><defs><linearGradient id="${id}" x1="0" y1="0" x2="0" y2="1"><stop offset="0" stop-color="#0057B8" stop-opacity=".22"/><stop offset="1" stop-color="#0057B8" stop-opacity="0"/></linearGradient></defs><path d="${d} L${w-p} ${h} L${p} ${h} Z" fill="url(#${id})"/><path d="${d}" fill="none" stroke="#0057B8" stroke-width="2" stroke-linecap="round" stroke-linejoin="round" vector-effect="non-scaling-stroke"/><circle cx="${e[0]}" cy="${e[1]}" r="3" fill="#FFC72C" stroke="#0057B8" stroke-width="1.5"/></svg>`;}
function kpiHtml(k){let dh=k.cmp===undefined?'':'<span class="pill n">–</span>';
  if(ok(k.cur)&&ok(k.cmp)){let dv,tx;if(k.dt==='rel'){dv=(k.cur/k.cmp-1)*100;tx=(dv>=0?'▲ ':'▼ ')+nf1.format(Math.abs(dv))+'%';}else if(k.dt==='pp'){dv=k.cur-k.cmp;tx=(dv>=0?'▲ ':'▼ ')+nf2.format(Math.abs(dv))+' pp';}else{dv=k.cur-k.cmp;tx=(dv>=0?'▲ ':'▼ ')+nf0.format(Math.abs(dv));}
    const cls=Math.abs(dv)<1e-9||!k.gd?'n':((dv>0)===(k.gd>0)?'g':'b');dh=`<span class="pill ${cls}">${tx}</span>`;}
  return `<div class="kpi ${k.hi?'hi':''}"><div class="l">${k.l}</div><div class="v">${k.fmt(k.cur)}</div><div class="dl">${dh}<span>${k.sub||''}</span></div>${k.spark?sparkSvg(k.spark):''}</div>`;}
function multiSelect(host,key,labels,S,onChange,name){
  const w=document.createElement('div');w.className='ms';
  w.innerHTML=`<button class="ms-b"><b>${name}</b><span>Tümü</span></button><div class="ms-p"><div class="act"><a data-a="all">Tümünü seç</a><a data-a="none">Temizle</a></div>`+labels.map((n,i)=>`<label><input type="checkbox" value="${i}" checked>${n}</label>`).join('')+'</div>';
  host.appendChild(w);const b=w.querySelector('.ms-b'),bx=[...w.querySelectorAll('input')];
  const ap=()=>{const sel=bx.filter(x=>x.checked).map(x=>+x.value);
    if(sel.length===bx.length||!sel.length){S.f[key].clear();b.classList.remove('on');b.querySelector('span').textContent='Tümü';if(!sel.length)bx.forEach(x=>x.checked=true);}
    else{S.f[key]=new Set(sel);b.classList.add('on');b.querySelector('span').textContent=sel.length<=2?sel.map(i=>labels[i]).join(', '):sel.length+' seçili';}onChange();};
  bx.forEach(x=>x.onchange=ap);w.querySelectorAll('.act a').forEach(a=>a.onclick=()=>{bx.forEach(x=>x.checked=a.dataset.a==='all');ap();});
  b.onclick=e=>{e.stopPropagation();const o=w.classList.contains('open');document.querySelectorAll('.ms').forEach(x=>x.classList.remove('open'));if(!o)w.classList.add('open');};
  w.querySelector('.ms-p').onclick=e=>e.stopPropagation();
  return {reset(){bx.forEach(x=>x.checked=true);b.classList.remove('on');b.querySelector('span').textContent='Tümü';}};}
document.addEventListener('click',()=>document.querySelectorAll('.ms').forEach(x=>x.classList.remove('open')));
function cmpBars(id,labels,cur,cmp,curName,cmpName,o={}){const h=o.h,tr=[];
  const mk=(nm,vals,col,lab)=>{const b={type:'bar',name:nm,marker:{color:col,line:{width:0}},text:vals.map(v=>lab&&ok(v)?(o.txt?o.txt(v):nf1.format(v)):''),textposition:'outside',cliponaxis:false,textfont:{size:11,color:C.mut},
      hovertemplate:'%{'+(h?'y':'x')+'}<br><b>%{'+(h?'x':'y')+':'+(o.hover||',.1f')+'}</b>'+(o.suf||'')+'<extra>'+nm+'</extra>'};if(h){b.y=labels;b.x=vals;b.orientation='h';}else{b.x=labels;b.y=vals;}return b;};
  if(cmp)tr.push(mk(cmpName,cmp,C.cmp,false));tr.push(mk(curName,cur,C.blue,true));
  const lay={barmode:'group',bargap:.3,bargroupgap:.06,margin:{l:h?96:50,r:h?46:12,t:30,b:38}};
  if(h){lay.yaxis={autorange:'reversed',automargin:true};lay.xaxis={rangemode:'tozero',gridcolor:C.grid};}else lay.yaxis={rangemode:'tozero'};
  if(o.lay)merge(lay,o.lay);plot(id,tr,lay);}
function donut(id,labels,vals,colors,center){plot(id,[{type:'pie',labels,values:vals,hole:.62,sort:false,marker:{colors,line:{color:'#fff',width:2}},textinfo:'percent',textfont:{size:11,color:'#fff'},hovertemplate:'%{label}<br><b>%{value:,.0f}</b> (%{percent})<extra></extra>'}],
  {margin:{l:4,r:4,t:4,b:4},showlegend:true,legend:{orientation:'v',x:1.0,y:.5,font:{size:11.5}},annotations:[{text:`<b>${center[0]}</b><br><span style="font-size:11px;color:#5E6E8C">${center[1]}</span>`,showarrow:false,font:{size:18,color:C.navy},x:.2,y:.5,xref:'paper'}]});}
function sortTable(host,cols,rows,state,render){const k=state.k,d=state.d;rows.sort((a,b)=>{const x=a[k],y=b[k];const v=typeof x==='string'?x.localeCompare(y,'tr'):((ok(x)?x:-1e18)-(ok(y)?y:-1e18));return v*d;});
  host.innerHTML='<table><thead><tr>'+cols.map(c=>`<th data-k="${c.k}" class="${k===c.k?'sorted':''}">${c.h}</th>`).join('')+'</tr></thead><tbody>'+rows.map(r=>'<tr>'+cols.map(c=>`<td class="${c.cls?c.cls(r):''}">${c.f(r)}</td>`).join('')+'</tr>').join('')+'</tbody></table>';
  host.querySelectorAll('th').forEach(th=>th.onclick=()=>{const kk=th.dataset.k;state.d=state.k===kk?-state.d:(kk==='nm'?1:-1);state.k=kk;render();});}
const sel=(el,opts,val)=>{el.innerHTML=opts.map(o=>`<option value="${o[0]}" ${String(o[0])===String(val)?'selected':''}>${o[1]}</option>`).join('');};
const delta=(v,inv,fmt)=>!ok(v)?'<span class="z">–</span>':`<span class="${(v>0)===!inv?'up':'dn'}">${v>0?'▲':'▼'} ${fmt(Math.abs(v))}</span>`;
const RENDER={},DIRTY={};let PAGE='p1';
function setPage(p){PAGE=p;document.querySelectorAll('.nb').forEach(b=>b.classList.toggle('on',b.dataset.p===p));document.querySelectorAll('.page').forEach(s=>s.classList.toggle('on',s.id===p));history.replaceState(null,'','#'+p);window.scrollTo(0,0);paint();
  setTimeout(()=>document.querySelectorAll('.page.on .ch').forEach(e=>{try{Plotly.Plots.resize(e)}catch(_){}}),80);}
function paint(){const r=RENDER[PAGE];if(r)r();}
document.querySelectorAll('.nb').forEach(b=>b.onclick=()=>setPage(b.dataset.p));
$('segName').textContent=META.segment;$('genAt').textContent='Üretim '+META.uretim;if(!META.sentetik)$('synTag').remove();

/* ═══════════════ ORTAK VERİ ERİŞİMİ ═══════════════ */
const DIMS=META.dims,ND=DIMS.length,DI={};DIMS.forEach((d,i)=>DI[d.key]=i);
const MET=META.metrics,MJ={};MET.forEach((m,j)=>MJ[m.key]=j);const SHARE=new Set(['comp','po3','pi12']),IV=META.ilgi,NI=IV.length;
const SK=new Map();{const S=D1.stock;for(let i=0;i<S.t.length;i++)SK.set((S.t[i]*32+S.d[i])*16+S.c[i],{n:S.n[i],s:S.s[i],q:S.q[i],i:S.i[i],u:S.u[i]});}
const recT=t=>SK.get((t*32)*16),rec=(t,di,c)=>SK.get((t*32+di+1)*16+c);
const hasD=k=>k in DI,hasM=k=>MJ[k]!=null&&MET[MJ[k]].ok,dl=di=>DIMS[di].labels,dnm=di=>DIMS[di].name;
const mean=(r,k)=>{const j=MJ[k];return r&&r.n?(SHARE.has(k)?100:1)*r.s[j]/r.n:NaN;};
const sdev=(r,k)=>{const j=MJ[k];if(!r||r.n<2)return NaN;const m=r.s[j]/r.n,v=Math.max(0,r.q[j]/r.n-m*m);return Math.sqrt(v)*(SHARE.has(k)?100:1);};
function fmtM(k,v){if(!ok(v))return '–';const u=MET[MJ[k]].unit;return u==='₺'?fTL(v):u==='GB'?nf1.format(v)+' GB':u==='dk'?nf0.format(v)+' dk':u==='ay'?nf1.format(v)+' ay':u==='yıl'?nf1.format(v)+' yaş':k==='arpur'?(v>0?'+':'')+nf1.format(v)+'%':nf1.format(v)+'%';}
const shareDim=(t,di,c)=>{const r=rec(t,di,c),T=recT(t);return r&&T?100*r.n/T.n:NaN;};
const intAvg=(r,j)=>r&&r.n?r.i[j]/r.n:NaN,intPen=(r,j)=>r&&r.n?100*r.u[j]/r.n:NaN;
const card=(d)=>`<div class="card ${d.cls||'c6'}"><h3>${d.t}</h3><div class="d">${d.d||''}</div>${d.tbl?`<div id="${d.id}" style="overflow-x:auto"></div>`:`<div class="ch ${d.h||''}" id="${d.id}" ${d.st?`style="${d.st}"`:''}></div>`}</div>`;
const mkCards=(host,defs)=>{$(host).innerHTML=defs.filter(d=>d.ok!==false).map(card).join('');};
const barLayout=(labs,h)=>h?{yaxis:{autorange:'reversed',automargin:true},xaxis:{ticksuffix:'%',gridcolor:C.grid,rangemode:'tozero'},margin:{l:130,r:60,t:10,b:30}}:{yaxis:{ticksuffix:'%',rangemode:'tozero'},xaxis:{tickfont:{size:10.5}},margin:{l:46,r:10,t:10,b:labs.some(l=>l.length>9)?70:40}};
function distChart(id,di,t,color){const labs=dl(di),T=recT(t),vals=labs.map((_,c)=>shareDim(t,di,c)),ns=labs.map((_,c)=>(rec(t,di,c)||{n:0}).n),h=labs.length>6;
  plot(id,[{type:'bar',orientation:h?'h':'v',[h?'y':'x']:labs,[h?'x':'y']:vals,marker:{color:color||C.blue},text:vals.map((v,i)=>fP1(v)),textposition:'outside',cliponaxis:false,textfont:{size:11,color:C.mut},customdata:ns,hovertemplate:'%{'+(h?'y':'x')+'}<br><b>%{customdata:,.0f}</b> müşteri (%{'+(h?'x':'y')+':.1f}%)<extra></extra>'}],Object.assign({showlegend:false},barLayout(labs,h)));}
function shareArpuChart(id,di,t){const labs=dl(di),vs=labs.map((_,c)=>shareDim(t,di,c)),ar=labs.map((_,c)=>mean(rec(t,di,c),'arpu')),rg=Math.max(...vs.filter(ok))*1.3;
  plot(id,[{type:'bar',x:labs,y:vs,name:'Müşteri payı',marker:{color:rgba(C.blue,.85)},text:vs.map(v=>fP1(v)),textposition:'outside',cliponaxis:false,textfont:{size:10.5,color:C.mut},hovertemplate:'%{x}<br>Pay <b>%{y:.1f}%</b><extra></extra>'},
    {type:'scatter',mode:'lines+markers',x:labs,y:ar,name:'ARPU (₺)',yaxis:'y2',line:{color:C.yellow,width:3},marker:{size:8,color:C.yellow,line:{color:C.navy,width:1.5}},hovertemplate:'%{x}<br>ARPU <b>₺%{y:,.0f}</b><extra></extra>'}],
    {yaxis:{ticksuffix:'%',range:[0,rg]},yaxis2:{overlaying:'y',side:'right',tickprefix:'₺',showgrid:false,tickfont:{color:C.mut},rangemode:'tozero'},xaxis:{tickfont:{size:10.5}},margin:{l:46,r:54,t:30,b:labs.some(l=>l.length>9)?72:40}});}
const heatCross=(cr,t,valFn,fmt,cs,id,ttl)=>{const a=cr.a,b=cr.b,na=dl(a).length,nb=dl(b).length,z=dl(a).map((_,i)=>dl(b).map((_,j)=>valFn(cr,i*nb+j)));
  plot(id,[{type:'heatmap',z,x:dl(b),y:dl(a),colorscale:cs,xgap:3,ygap:3,showscale:false,texttemplate:fmt,textfont:{size:11},hovertemplate:'%{y} · %{x}<br><b>%{z:,.1f}</b><extra></extra>'}],{showlegend:false,yaxis:{autorange:'reversed'},margin:{l:96,r:10,t:8,b:56},xaxis:{tickfont:{size:10.5}}});};
const crossOf=(a,b)=>D1.cross.find(c=>c.a===DI[a]&&c.b===DI[b]);
const BLUES=[[0,'#EAF2FF'],[.5,'#4DA3FF'],[1,'#0A2A5E']];
const T1=NT-1,S1={tab:'g',aDim:0,iMet:'avg',iDim:0,rDim:0},D1T={};
$('p1sub').innerHTML=`Güncel durum: <b>${ML(T1)}</b> · karşılaştırma içermez; geçmiş aylarla kıyas için "Karşılaştırma" sayfasına gidin.`;$('nb1').textContent=ML(T1)+' güncel durumu';

/* ── Genel görünüm ── */
function p1g(){const T=recT(T1),t=T1;if(!$('g_grid').dataset.b){
  mkCards('g_grid',[{id:'g_hist',t:'Mevcut müşterilerin ARPU seyri',d:'Bu ayın portföyündeki müşterilerin son 6 aydaki ortalama ARPU\'su (ARPU_L6M … ARPU_TL)',cls:'c6'},{id:'g_b1',t:'ARPU dağılımı',d:'P10 – P90 aralığı, kutu: P25–P75, çizgi: medyan',cls:'c6',h:'s'},
    {id:'g_b2',t:'Data kullanımı (GB)',d:'P10 – P90',cls:'c4',h:'s'},{id:'g_b3',t:'Ses kullanımı (dk)',d:'P10 – P90',cls:'c4',h:'s'},{id:'g_b4',t:'Kıdem (ay)',d:'P10 – P90',cls:'c4',h:'s'},{id:'g_ins',t:'Bu ayın öne çıkanları',d:'Portföyden otomatik çıkarılan özet bilgiler',cls:'c12',tbl:1}]);$('g_grid').dataset.b=1;}
  const kp=[{l:'Aktif müşteri',cur:T.n,fmt:fInt,hi:1,sub:'portföyde'},{l:'ARPU',cur:mean(T,'arpu'),fmt:fTL,hi:1,sub:'müşteri başı / ay'},{l:'Toplam gelir',cur:T.s[MJ.arpu],fmt:v=>ok(v)?'₺'+nf1.format(v/1e6)+' Mn':'–',sub:'aylık'},
    {l:'Müşteri başı data',cur:mean(T,'gb'),fmt:v=>fmtM('gb',v),sub:'aylık ortalama'},{l:'Müşteri başı ses',cur:mean(T,'mou'),fmt:v=>fmtM('mou',v),sub:'aylık ortalama'},{l:'Ortalama kıdem',cur:mean(T,'ten'),fmt:v=>fmtM('ten',v)},
    hasM('age')?{l:'Ortalama yaş',cur:mean(T,'age'),fmt:v=>fmtM('age',v)}:null,hasD('gen')?{l:'Kadın payı',cur:shareDim(t,DI.gen,0),fmt:fP1,sub:'erkek '+fP1(shareDim(t,DI.gen,1))}:null,hasD('con')?{l:dl(DI.con)[0]+' payı',cur:shareDim(t,DI.con,0),fmt:fP1}:null,
    {l:'Şikayeti olan',cur:mean(T,'comp'),fmt:fP2,hi:1},{l:'Son 3 ay port-out olan',cur:mean(T,'po3'),fmt:fP2},hasD('at')?{l:'ARPU düşüşündeki müşteri',cur:shareDim(t,DI.at,0),fmt:fP1,sub:'son 3 ay ortalamasının %10 altı'}:null].filter(Boolean);
  $('g_kpis').innerHTML=kp.map(kpiHtml).join('');
  const AH=D1.arpuHist.filter(x=>ok(x.ort));plot('g_hist',[{type:'scatter',mode:'lines+markers',x:AH.map(x=>x.lag==='Bu ay'?MS(T1):x.lag),y:AH.map(x=>x.ort),line:{color:C.blue,width:3.5,shape:'spline'},marker:{size:9,color:C.yellow,line:{color:C.blue,width:2}},fill:'tozeroy',fillcolor:rgba(C.blue,.07),text:AH.map(x=>fTL(x.ort)),textposition:'top center',mode:'lines+markers+text',textfont:{size:11,color:C.navy},hovertemplate:'%{x}<br>Ort. ARPU <b>₺%{y:,.0f}</b><extra></extra>'}],
    {showlegend:false,xaxis:{title:{text:'Geriye doğru ay (L6M = 6 ay önce)'},range:[-.4,AH.length-.6]},yaxis:{tickprefix:'₺',range:[Math.min(...AH.map(x=>x.ort))*.85,Math.max(...AH.map(x=>x.ort))*1.1]},margin:{l:62,r:36,t:30,b:50}});
  const Q=D1.quant[t],box=(id,k,col)=>{const q=Q[k];if(!q)return;plot(id,[{type:'box',orientation:'h',y:[MET[MJ[k]].name],lowerfence:[q[0]],q1:[q[1]],median:[q[2]],q3:[q[3]],upperfence:[q[4]],marker:{color:col},fillcolor:rgba(col,.25),line:{color:col,width:2.5},boxpoints:false,hovertemplate:'P10 '+q[0]+'<br>P25 '+q[1]+'<br>Medyan <b>'+q[2]+'</b><br>P75 '+q[3]+'<br>P90 '+q[4]+'<extra></extra>'}],{showlegend:false,yaxis:{showticklabels:false},margin:{l:20,r:20,t:10,b:36}});};
  box('g_b1','arpu',C.blue);box('g_b2','gb',C.sky);box('g_b3','mou','#7B6CC4');box('g_b4','ten',C.good);
  const ins=[],top=di=>{const v=dl(di).map((_,c)=>[shareDim(t,di,c),c]).sort((a,b)=>b[0]-a[0]);return v[0];};
  const add=(i,x,k='n')=>ins.push(`<div class="ins ${k}"><div class="i">${i}</div><div><div>${x}</div></div></div>`);
  add('◎',`<b>${fInt(T.n)}</b> aktif müşterinin ortalama ARPU'su <b>${fTL(mean(T,'arpu'))}</b>, toplam aylık geliri <b>₺${nf1.format(T.s[MJ.arpu]/1e6)} Mn</b>; müşteri başı <b>${nf1.format(mean(T,'gb'))} GB</b> data ve <b>${nf0.format(mean(T,'mou'))} dk</b> ses kullanımı.`);
  if(hasD('city')){const [s,c]=top(DI.city);add('⌖',`En büyük şehir <b>${dl(DI.city)[c]}</b> (${fP1(s)}); ilk 3 şehir portföyün <b>${fP1(dl(DI.city).map((_,c)=>shareDim(t,DI.city,c)).sort((a,b)=>b-a).slice(0,3).reduce((a,b)=>a+b,0))}</b>'ini oluşturuyor.`);}
  if(hasD('tar')){const [s,c]=top(DI.tar);add('▤',`En yaygın tarife <b>${dl(DI.tar)[c]}</b> (${fP1(s)}), bu tarifenin ARPU'su <b>${fTL(mean(rec(t,DI.tar,c),'arpu'))}</b>.`);}
  if(hasD('dev')){const [s,c]=top(DI.dev);add('☎',`Cihaz pazarında lider marka <b>${dl(DI.dev)[c]}</b> (${fP1(s)}); ARPU'su <b>${fTL(mean(rec(t,DI.dev,c),'arpu'))}</b>.`);}
  if(hasD('dig')){const [s,c]=top(DI.dig);add('▦',`En büyük dijital segment <b>${dl(DI.dig)[c]}</b> (${fP1(s)}).`);}
  if(hasD('at'))add(shareDim(t,DI.at,0)>shareDim(t,DI.at,2)?'↘':'↗',`ARPU'su son 3 ay ortalamasına göre %10'dan fazla <b>düşen müşteri payı ${fP1(shareDim(t,DI.at,0))}</b>, <b>artan müşteri payı ${fP1(shareDim(t,DI.at,2))}</b>.`,shareDim(t,DI.at,0)>shareDim(t,DI.at,2)?'b':'g');
  if(hasD('mnp'))add('⇄',`Müşterilerin <b>${fP1(shareDim(t,DI.mnp,1))}</b>'inin geçmişte numara taşıma (MNP) hareketi var; son 3 ayda port-out yapmış olanlar <b>${fP2(mean(T,'po3'))}</b>.`,'b');
  $('g_ins').innerHTML=ins.join('');}

/* ── Müşteri profili ── */
function p1p(){const t=T1;if(!$('p_grid').dataset.b){const L=[],c=(k,tt,dd,cls,h)=>hasD(k)&&L.push({id:'p_'+k,t:tt,d:dd,cls:cls||'c6',h});
  c('age','Yaş dağılımı','Müşteri payı (%)');c('gen','Cinsiyet','Müşteri payı (%)','c3','s');c('at','ARPU eğilimi','Son 3 ay ortalamasına göre ARPU değişimi','c3','s');c('city','Şehir dağılımı','İlk 10 şehir + diğer','c6','t');c('ten','Kıdem dağılımı','Şirkette geçen süre');
  c('dev','Cihaz markası','Müşteri payı (%)','c6','t');c('bon','Bonus','Tanımlı bonus adı','c6');if(hasD('age')&&hasD('gen'))L.push({id:'p_x1',t:'Yaş × cinsiyet',d:'Müşteri adedi',cls:'c6',h:'s'});if(hasD('age')&&hasD('tar'))L.push({id:'p_x2',t:'Yaş × tarife',d:'Müşteri adedi',cls:'c6',h:'s'});
  mkCards('p_grid',L);$('p_grid').dataset.b=1;}
  ['age','ten','dev','city','bon'].forEach(k=>hasD(k)&&distChart('p_'+k,DI[k],t));
  ['gen','at'].forEach(k=>{if(!hasD(k))return;const labs=dl(DI[k]);plot('p_'+k,[{type:'pie',labels:labs,values:labs.map((_,c)=>(rec(t,DI[k],c)||{n:0}).n),hole:.6,sort:false,marker:{colors:k==='at'?[C.bad,C.cmp,C.good]:[C.yellow,C.blue],line:{color:'#fff',width:2}},textinfo:'percent',textfont:{size:11,color:'#fff'},hovertemplate:'%{label}<br><b>%{value:,.0f}</b> (%{percent})<extra></extra>'}],{margin:{l:4,r:4,t:4,b:4},showlegend:true,legend:{orientation:'h',y:-.05,font:{size:10.5}}});});
  const x1=crossOf('age','gen');if(x1)heatCross(x1,t,(c,i)=>c.n[i],'%{z:,.0f}',BLUES,'p_x1');const x2=crossOf('age','tar');if(x2)heatCross(x2,t,(c,i)=>c.n[i],'%{z:,.0f}',BLUES,'p_x2');}

/* ── ARPU ve kullanım ── */
function p1a(){const t=T1,di=S1.aDim,labs=dl(di);if(!$('a_grid').dataset.b){mkCards('a_grid',[{id:'a_1',t:'ARPU ve müşteri payı',d:'Çubuk: müşteri payı · çizgi: ortalama ARPU',cls:'c6'},{id:'a_2',t:'Data ve ses kullanımı',d:'Çubuk: müşteri başı GB · çizgi: müşteri başı dakika',cls:'c6'},
    {id:'a_3',t:'Gelir payı ve müşteri payı',d:'Hangi grup müşteri sayısından fazla gelir getiriyor?',cls:'c6'},{id:'a_4',t:'Tarife ücreti ve ARPU',d:'Çubuk: tarife ücreti · çizgi: ARPU (fark = tarife dışı kullanım ve ek hizmet)',cls:'c6'},
    {id:'a_5',t:'ARPU bandı dağılımı',d:'Bantlar bu ayın portföyünden (P20/P40/P60/P80) belirlenir',cls:'c4',ok:hasD('ab')},{id:'a_6',t:'Data kullanım bandı dağılımı',d:'',cls:'c4',ok:hasD('gbb')},{id:'a_7',t:'ARPU eğilimi',d:'Son 3 ay ortalamasına göre',cls:'c4',ok:hasD('at')},
    {id:'a_8',t:'ARPU ısı haritası · tarife × sözleşme',d:'₺',cls:'c6',h:'s',ok:!!crossOf('tar','con')},{id:'a_9',t:'ARPU ısı haritası · yaş × tarife',d:'₺',cls:'c6',h:'s',ok:!!crossOf('age','tar')}]);$('a_grid').dataset.b=1;}
  shareArpuChart('a_1',di,t);
  const gbv=labs.map((_,c)=>mean(rec(t,di,c),'gb')),mv=labs.map((_,c)=>mean(rec(t,di,c),'mou'));
  plot('a_2',[{type:'bar',x:labs,y:gbv,name:'GB / müşteri',marker:{color:C.sky},text:gbv.map(v=>nf1.format(v)),textposition:'outside',cliponaxis:false,textfont:{size:10.5,color:C.mut}},{type:'scatter',mode:'lines+markers',x:labs,y:mv,name:'dk / müşteri',yaxis:'y2',line:{color:'#7B6CC4',width:3},marker:{size:8}}],
    {yaxis:{rangemode:'tozero'},yaxis2:{overlaying:'y',side:'right',showgrid:false,tickfont:{color:C.mut},rangemode:'tozero'},xaxis:{tickfont:{size:10.5}},margin:{l:46,r:50,t:30,b:labs.some(l=>l.length>9)?72:40}});
  const cs=labs.map((_,c)=>shareDim(t,di,c)),T=recT(t),rs=labs.map((_,c)=>100*(rec(t,di,c)||{s:[0]}).s[MJ.arpu]/T.s[MJ.arpu]);
  plot('a_3',[{type:'bar',name:'Müşteri payı',x:labs,y:cs,marker:{color:C.cmp},text:cs.map(v=>nf1.format(v)+'%'),textposition:'outside',cliponaxis:false,textfont:{size:10.5,color:C.mut}},{type:'bar',name:'Gelir payı',x:labs,y:rs,marker:{color:C.blue},text:rs.map(v=>nf1.format(v)+'%'),textposition:'outside',cliponaxis:false,textfont:{size:10.5,color:C.mut}}],{barmode:'group',bargap:.3,yaxis:{ticksuffix:'%',rangemode:'tozero'},xaxis:{tickfont:{size:10.5}},margin:{l:46,r:10,t:30,b:labs.some(l=>l.length>9)?72:40}});
  const pr=labs.map((_,c)=>mean(rec(t,di,c),'price')),ar=labs.map((_,c)=>mean(rec(t,di,c),'arpu'));
  plot('a_4',[{type:'bar',x:labs,y:pr,name:'Tarife ücreti',marker:{color:C.cmp},text:pr.map(v=>fTL(v)),textposition:'outside',cliponaxis:false,textfont:{size:10.5,color:C.mut}},{type:'scatter',mode:'lines+markers',x:labs,y:ar,name:'ARPU',line:{color:C.yellow,width:3},marker:{size:8,color:C.yellow,line:{color:C.navy,width:1.5}}}],{yaxis:{tickprefix:'₺',rangemode:'tozero'},xaxis:{tickfont:{size:10.5}},margin:{l:56,r:10,t:30,b:labs.some(l=>l.length>9)?72:40}});
  if(hasD('ab'))distChart('a_5',DI.ab,t);if(hasD('gbb'))distChart('a_6',DI.gbb,t,C.sky);
  if(hasD('at')){const l=dl(DI.at);plot('a_7',[{type:'pie',labels:l,values:l.map((_,c)=>(rec(t,DI.at,c)||{n:0}).n),hole:.6,sort:false,marker:{colors:[C.bad,C.cmp,C.good],line:{color:'#fff',width:2}},textinfo:'percent',textfont:{size:11,color:'#fff'}}],{margin:{l:4,r:4,t:4,b:4},legend:{orientation:'h',y:-.05,font:{size:10}}});}
  const c1=crossOf('tar','con');if(c1)heatCross(c1,t,(c,i)=>c.n[i]?c.arpu[i]/c.n[i]:NaN,'₺%{z:,.0f}',BLUES,'a_8');const c2=crossOf('age','tar');if(c2)heatCross(c2,t,(c,i)=>c.n[i]?c.arpu[i]/c.n[i]:NaN,'₺%{z:,.0f}',BLUES,'a_9');}

/* ── Tarife, cihaz ve dijital segment ── */
function p1t(){const t=T1,ks=['tar','dig','dev','con','chg','rah'].filter(hasD);if(!$('t_grid').dataset.b){mkCards('t_grid',ks.map(k=>({id:'t_'+k,t:dnm(DI[k])+' · müşteri payı ve ARPU',d:'Çubuk: müşteri payı · çizgi: ARPU',cls:'c6'})).concat(crossOf('dig','dev')?[{id:'t_x',t:'Dijital segment × cihaz markası',d:'Müşteri adedi',cls:'c12',h:'s'}]:[]));$('t_grid').dataset.b=1;}
  ks.forEach(k=>shareArpuChart('t_'+k,DI[k],t));const x=crossOf('dig','dev');if(x)heatCross(x,t,(c,i)=>c.n[i],'%{z:,.0f}',BLUES,'t_x');}

/* ── Dijital ilgi alanları ── */
function p1i(){const t=T1,T=recT(t),j=S1.iMet==='avg'?intAvg:intPen,unit=S1.iMet==='avg'?'':'%';if(!$('i_grid').dataset.b){mkCards('i_grid',[{id:'i_bar',t:'Hangi ilgi alanında ne kadar arama yapılıyor?',d:'Müşterinin ilgili sitelerde yaptığı arama adedi',cls:'c5',st:'height:700px'},{id:'i_heat',t:'İlgi alanı × kırılım indeksi',d:'100 = portföy ortalaması · 100\'ün üstü o grubun ilgiyi daha yoğun gösterdiğini belirtir',cls:'c7',st:'height:700px'}]);$('i_grid').dataset.b=1;}
  const ord=IV.map((_,k)=>k).sort((a,b)=>j(T,b)-j(T,a)),lab=ord.map(k=>IV[k]),fm=v=>S1.iMet==='avg'?nf1.format(v):nf1.format(v)+'%';
  plot('i_bar',[{type:'bar',orientation:'h',y:lab,x:ord.map(k=>j(T,k)),marker:{color:C.blue},text:ord.map(k=>fm(j(T,k))),textposition:'outside',cliponaxis:false,textfont:{size:11,color:C.mut},hovertemplate:'%{y}<br><b>%{x:.2f}</b>'+unit+'<extra></extra>'}],{showlegend:false,yaxis:{autorange:'reversed',automargin:true},xaxis:{rangemode:'tozero',gridcolor:C.grid,ticksuffix:unit},margin:{l:110,r:50,t:10,b:30}});
  const di=S1.iDim,labs=dl(di),z=ord.map(k=>labs.map((_,c)=>{const b=j(T,k),v=j(rec(t,di,c),k);return ok(v)&&b>0?100*v/b:NaN;}));
  plot('i_heat',[{type:'heatmap',z,x:labs,y:lab,zmin:40,zmax:200,colorscale:[[0,'#4DA3FF'],[.3,'#EAF2FF'],[.45,'#FFFFFF'],[.65,'#FFE08A'],[1,'#E5484D']],xgap:2,ygap:2,showscale:true,colorbar:{thickness:10,len:.6,tickfont:{size:10,color:C.mut}},texttemplate:'%{z:.0f}',textfont:{size:11},hovertemplate:'%{y} · %{x}<br>İndeks <b>%{z:.0f}</b><extra></extra>'}],{showlegend:false,yaxis:{autorange:'reversed'},xaxis:{side:'top',tickfont:{size:10.5}},margin:{l:110,r:10,t:50,b:6}});}

/* ── Risk sinyalleri ve karne ── */
const TB1={k:'n',d:-1};
function p1r(){const t=T1,di=S1.rDim,labs=dl(di);if(!$('r_grid').dataset.b){mkCards('r_grid',[{id:'r_1',t:'Risk sinyalleri · seçili kırılıma göre',d:'Şikayeti olan, son 3 ayda port-out yapan ve son 12 ayda port-in yapan müşteri oranları (%)',cls:'c12'},{id:'r_tbl',t:'Kırılım karnesi',d:'Seçili kırılımda tüm değişkenlerin ortalamaları · başlığa tıklayarak sıralayın',cls:'c12',tbl:1}]);$('r_grid').dataset.b=1;}
  const tr=[['comp','Şikayeti olan',C.bad],['po3','Son 3 ay port-out',C.yellow],['pi12','Son 12 ay port-in',C.blue]].filter(x=>hasM(x[0])).map(x=>({type:'bar',name:x[1],x:labs,y:labs.map((_,c)=>mean(rec(t,di,c),x[0])),marker:{color:x[2]},text:labs.map((_,c)=>nf1.format(mean(rec(t,di,c),x[0]))+'%'),textposition:'outside',cliponaxis:false,textfont:{size:10.5,color:C.mut}}));
  plot('r_1',tr,{barmode:'group',bargap:.28,yaxis:{ticksuffix:'%',rangemode:'tozero'},xaxis:{tickfont:{size:10.5}},margin:{l:46,r:10,t:34,b:labs.some(l=>l.length>9)?72:40}});
  const cols=[['nm','Grup'],['n','Müşteri'],['sh','Pay']].concat(MET.filter(m=>m.ok).map(m=>[m.key,m.name]));
  const rows=labs.map((nm,c)=>{const r=rec(t,di,c),o={nm,n:r?r.n:0,sh:shareDim(t,di,c)};MET.forEach(m=>o[m.key]=mean(r,m.key));return o;});
  sortTable($('r_tbl'),cols.map(([k,h])=>({k,h,f:r=>k==='nm'?`<b>${r.nm}</b>`:k==='n'?fInt(r.n):k==='sh'?fP1(r.sh):fmtM(k,r[k])})),rows,TB1,p1r);}

const R1={g:p1g,p:p1p,a:p1a,t:p1t,i:p1i,r:p1r};
function paint1(){if(!D1T[S1.tab]){D1T[S1.tab]=1;R1[S1.tab]();}}
RENDER.p1=paint1;
function init1(){
  const dimOpts=DIMS.map((d,i)=>[i,d.name]);sel($('a_dim'),dimOpts,S1.aDim);sel($('i_dim'),dimOpts,S1.iDim);sel($('r_dim'),dimOpts,S1.rDim);
  $('a_dim').onchange=e=>{S1.aDim=+e.target.value;p1a();};$('i_dim').onchange=e=>{S1.iDim=+e.target.value;p1i();};$('r_dim').onchange=e=>{S1.rDim=+e.target.value;p1r();};
  document.querySelectorAll('#i_met button').forEach(b=>b.onclick=()=>{S1.iMet=b.dataset.v;document.querySelectorAll('#i_met button').forEach(x=>x.classList.toggle('on',x===b));p1i();});
  document.querySelectorAll('#p1tabs button').forEach(b=>b.onclick=()=>{S1.tab=b.dataset.t;document.querySelectorAll('#p1tabs button').forEach(x=>x.classList.toggle('on',x===b));document.querySelectorAll('#p1 .tab').forEach(s=>s.classList.toggle('on',s.id==='p1-'+S1.tab));paint1();
    setTimeout(()=>document.querySelectorAll('#p1-'+S1.tab+' .ch').forEach(e=>{try{Plotly.Plots.resize(e)}catch(_){}}),60);});}
init1();

/* ═══════════════ SAYFA 2 · KARŞILAŞTIRMA ═══════════════ */
const S2={a:Math.max(0,NT-3),b:NT-1,d:0,met:'arpu',oDim:0,tab:'g',al:{c:META.alert.adet,o:META.alert.ort,m:META.alert.min,cnt:true,mean:true,int:true}},D2T={};
const PAIR=new Map();D1.pairs.forEach(p=>PAIR.set(p.a+'_'+p.b+'_'+p.d,p.m));const AHM=new Map();D1.ahist.forEach(h=>AHM.set(h.a+'_'+h.b,h));
const RISK=new Set(['comp','po3','pi12']);
const relp=(b,a)=>a?(b/a-1)*100:NaN,An=()=>MS(S2.a),Bn=()=>MS(S2.b);
function resolve2(){$('p2s').innerHTML=`Baz: <b>${ML(S2.a)}</b> → Karşılaştırılan: <b>${ML(S2.b)}</b> (${S2.b-S2.a} ay fark) · Kırılım: <b>${dnm(S2.d)}</b>`;}
const mkGroupList=()=>[{di:-1,name:'Toplam',labs:['Tüm müşteriler']}].concat(DIMS.map((d,i)=>({di:i,name:d.name,labs:d.labels})));
const recG=(t,g,c)=>g.di<0?recT(t):rec(t,g.di,c);

/* ── Genel fark ── */
function p2g(){const a=S2.a,b=S2.b,TA=recT(a),TB=recT(b);if(!$('c_grid').dataset.b){mkCards('c_grid',[{id:'c_1',t:'Ortalamalardaki değişim (%)',d:'Seçilen iki ay arasındaki göreli değişim',cls:'c6'},{id:'c_2',t:'Oranlardaki değişim (puan)',d:'Şikayet, port-out, port-in oranları ve ARPU değişimi · puan farkı',cls:'c6'},
    {id:'c_3',t:'Adedi en çok değişen gruplar',d:'Tüm kırılımlar taranır; en az grup büyüklüğü uyarı eşiğiyle aynıdır',cls:'c12',h:'k'}]);$('c_grid').dataset.b=1;}
  const K=[{k:'n',l:'Aktif müşteri',fmt:fInt,A:TA.n,B:TB.n,dt:'rel',gd:1,hi:1},{k:'rev',l:'Toplam gelir',fmt:v=>'₺'+nf1.format(v/1e6)+' Mn',A:TA.s[MJ.arpu],B:TB.s[MJ.arpu],dt:'rel',gd:1}]
    .concat([['arpu',1],['gb',0],['mou',0],['price',0],['ten',1],['age',0]].filter(x=>hasM(x[0])).map(([k,gd])=>({k,l:MET[MJ[k]].name,fmt:v=>fmtM(k,v),A:mean(TA,k),B:mean(TB,k),dt:'rel',gd})))
    .concat([['comp',-1],['po3',-1],['pi12',-1]].filter(x=>hasM(x[0])).map(([k,gd])=>({k,l:MET[MJ[k]].name,fmt:v=>fmtM(k,v),A:mean(TA,k),B:mean(TB,k),dt:'pp',gd})));
  $('c_kpis').innerHTML=K.map((x,i)=>kpiHtml({l:x.l,cur:x.B,cmp:x.A,fmt:x.fmt,dt:x.dt,gd:x.gd,hi:i<2,sub:`${x.fmt(x.A)} → ${x.fmt(x.B)}`})).join('');
  const rl=['arpu','gb','mou','price','ten','age'].filter(hasM),pp=['comp','po3','pi12','arpur'].filter(hasM);
  const barDelta=(id,ks,fn,unit)=>{const v=ks.map(fn);plot(id,[{type:'bar',orientation:'h',y:ks.map(k=>MET[MJ[k]].name),x:v,marker:{color:v.map(x=>x>=0?C.good:C.bad)},text:v.map(x=>(x>0?'+':'')+nf2.format(x)+unit),textposition:'outside',cliponaxis:false,textfont:{size:11,color:C.mut},hovertemplate:'%{y}<br><b>%{x:+.2f}</b>'+unit+'<extra></extra>'}],
    {showlegend:false,yaxis:{autorange:'reversed',automargin:true},xaxis:{gridcolor:C.grid,zeroline:true,zerolinecolor:C.navy,ticksuffix:unit},margin:{l:180,r:60,t:10,b:30}});};
  barDelta('c_1',rl,k=>relp(mean(TB,k),mean(TA,k)),'%');
  barDelta('c_2',pp,k=>mean(TB,k)-mean(TA,k),' puan');
  const L=[];mkGroupList().forEach(g=>{if(g.di<0||['ab','gbb'].includes(DIMS[g.di].key))return;g.labs.forEach((c,ci)=>{const x=recG(a,g,ci),y=recG(b,g,ci);if(x&&y&&x.n>=S2.al.m&&y.n>=S2.al.m)L.push({lbl:g.name+' · '+c,rel:relp(y.n,x.n),na:x.n,nb:y.n});});});
  L.sort((p,q)=>Math.abs(q.rel)-Math.abs(p.rel));const top=L.slice(0,16);
  plot('c_3',[{type:'bar',orientation:'h',y:top.map(x=>x.lbl),x:top.map(x=>x.rel),marker:{color:top.map(x=>x.rel>=0?C.good:C.bad)},text:top.map(x=>`${x.rel>0?'+':''}${nf1.format(x.rel)}%  (${fInt(x.na)} → ${fInt(x.nb)})`),textposition:'outside',cliponaxis:false,textfont:{size:11,color:C.mut},hovertemplate:'%{y}<br><b>%{x:+.1f}%</b><extra></extra>'}],
    {showlegend:false,yaxis:{autorange:'reversed',automargin:true},xaxis:{gridcolor:C.grid,zeroline:true,zerolinecolor:C.navy,ticksuffix:'%'},margin:{l:250,r:130,t:10,b:30}});}

/* ── Dağılım değişimi ── */
const TB2={k:'dn',d:-1};
function p2d(){const a=S2.a,b=S2.b,di=S2.d,labs=dl(di),TA=recT(a),TB=recT(b);if(!$('d_grid').dataset.b){mkCards('d_grid',[{id:'d_1',t:'Dağılım: A ve B ayı',d:'Müşteri payı (%)',cls:'c6'},{id:'d_2',t:'Payın değişimi (puan)',d:'Yeşil: payı artan · kırmızı: payı azalan',cls:'c6'},{id:'d_3',t:'Hangi kırılımda dağılım en çok değişti?',d:'Her kırılımda en büyük pay değişimi (puan, mutlak)',cls:'c5'},{id:'d_4',t:'Adet değişimi',d:'Gruplara göre müşteri adedi · başlığa tıklayarak sıralayın',cls:'c7',tbl:1}]);$('d_grid').dataset.b=1;}
  const sA=labs.map((_,c)=>100*((rec(a,di,c)||{n:0}).n)/TA.n),sB=labs.map((_,c)=>100*((rec(b,di,c)||{n:0}).n)/TB.n),h=labs.length>6;
  cmpBars('d_1',labs,sB,sA,Bn(),An(),{h,hover:'.1f',suf:'%',txt:v=>nf1.format(v)+'%'});
  const dp=sB.map((v,i)=>v-sA[i]);plot('d_2',[{type:'bar',orientation:h?'h':'v',[h?'y':'x']:labs,[h?'x':'y']:dp,marker:{color:dp.map(x=>x>=0?C.good:C.bad)},text:dp.map(x=>(x>0?'+':'')+nf2.format(x)),textposition:'outside',cliponaxis:false,textfont:{size:11,color:C.mut},hovertemplate:'%{'+(h?'y':'x')+'}<br><b>%{'+(h?'x':'y')+':+.2f}</b> puan<extra></extra>'}],
    Object.assign({showlegend:false},h?{yaxis:{autorange:'reversed',automargin:true},xaxis:{gridcolor:C.grid,zeroline:true,zerolinecolor:C.navy,ticksuffix:' p'},margin:{l:130,r:50,t:10,b:30}}:{yaxis:{gridcolor:C.grid,zeroline:true,zerolinecolor:C.navy,ticksuffix:' p'},xaxis:{tickfont:{size:10.5}},margin:{l:46,r:10,t:20,b:labs.some(l=>l.length>9)?70:40}}));
  const mx=DIMS.map((d,i)=>({n:d.name,v:Math.max(...d.labels.map((_,c)=>Math.abs(100*((rec(b,i,c)||{n:0}).n)/TB.n-100*((rec(a,i,c)||{n:0}).n)/TA.n)))})).sort((p,q)=>q.v-p.v);
  plot('d_3',[{type:'bar',orientation:'h',y:mx.map(x=>x.n),x:mx.map(x=>x.v),marker:{color:C.blue},text:mx.map(x=>nf2.format(x.v)),textposition:'outside',cliponaxis:false,textfont:{size:11,color:C.mut}}],{showlegend:false,yaxis:{autorange:'reversed',automargin:true},xaxis:{gridcolor:C.grid,ticksuffix:' p',rangemode:'tozero'},margin:{l:150,r:40,t:10,b:30}});
  const rows=labs.map((nm,c)=>{const x=rec(a,di,c)||{n:0},y=rec(b,di,c)||{n:0};return {nm,na:x.n,nb:y.n,dn:y.n-x.n,rel:relp(y.n,x.n),pa:sA[c],pb:sB[c],dp:sB[c]-sA[c]};});
  const cols=[['nm','Grup'],['na','A adedi'],['nb','B adedi'],['dn','Fark'],['rel','Fark %'],['pa','A payı'],['pb','B payı'],['dp','Pay farkı']];
  sortTable($('d_4'),cols.map(([k,hh])=>({k,h:hh,f:r=>k==='nm'?`<b>${r.nm}</b>`:k==='na'||k==='nb'?fInt(r[k]):k==='dn'?fS(r.dn):k==='rel'?delta(r.rel,false,v=>nf1.format(v)+'%'):k==='pa'||k==='pb'?fP1(r[k]):delta(r.dp,false,v=>nf2.format(v)+' p')})),rows,TB2,p2d);}

/* ── Ortalama değişimi ── */
function p2a(){const a=S2.a,b=S2.b,di=S2.d,labs=dl(di),ms=MET.filter(m=>m.ok).map(m=>m.key);if(!$('m_grid').dataset.b){mkCards('m_grid',[{id:'m_1',t:'Ortalama değişimi ısı haritası',d:'Satır grup, sütun değişken · hücre = B ayının A ayına göre değişimi (ortalama % ; oranlar için puan)',cls:'c12',st:'height:'+Math.max(300,labs.length*38+90)+'px'},{id:'m_2',t:'Seçili ölçüde grup ortalamaları',d:'A ve B ayı yan yana',cls:'c12'}]);$('m_grid').dataset.b=1;}
  $('m_1').style.height=Math.max(300,labs.length*38+90)+'px';
  const z=labs.map((_,c)=>ms.map(k=>{const x=rec(a,di,c),y=rec(b,di,c);if(!x||!y)return NaN;const mA=mean(x,k),mB=mean(y,k);return (SHARE.has(k)||k==='arpur')?mB-mA:relp(mB,mA);}));
  const tx=labs.map((_,c)=>ms.map((k,j)=>ok(z[c][j])?((z[c][j]>0?'+':'')+nf1.format(z[c][j])+((SHARE.has(k)||k==='arpur')?' p':'%')):''));
  plot('m_1',[{type:'heatmap',z,x:ms.map(k=>MET[MJ[k]].name),y:labs,zmin:-15,zmax:15,colorscale:[[0,'#E5484D'],[.5,'#FFFFFF'],[1,'#13A97B']],xgap:3,ygap:3,showscale:true,colorbar:{thickness:10,len:.6,tickfont:{size:10,color:C.mut}},text:tx,texttemplate:'%{text}',textfont:{size:11},hovertemplate:'%{y} · %{x}<br><b>%{text}</b><extra></extra>'}],{showlegend:false,yaxis:{autorange:'reversed'},xaxis:{side:'top',tickfont:{size:10.5}},margin:{l:150,r:10,t:70,b:6}});
  const k=S2.met;plot('m_2',[{type:'bar',name:An(),x:labs,y:labs.map((_,c)=>mean(rec(a,di,c),k)),marker:{color:C.cmp}},{type:'bar',name:Bn(),x:labs,y:labs.map((_,c)=>mean(rec(b,di,c),k)),marker:{color:C.blue},text:labs.map((_,c)=>fmtM(k,mean(rec(b,di,c),k))),textposition:'outside',cliponaxis:false,textfont:{size:10.5,color:C.mut}}],
    {barmode:'group',bargap:.3,yaxis:{rangemode:'tozero'},xaxis:{tickfont:{size:10.5}},margin:{l:56,r:10,t:30,b:labs.some(l=>l.length>9)?72:40}});}

/* ── Ortak müşteriler ve geçişler ── */
function p2o(){const a=S2.a,b=S2.b,h=AHM.get(a+'_'+b);if(!$('o_grid').dataset.b){mkCards('o_grid',[{id:'o_sum',t:'İki ayda da portföyde olan müşteriler',d:'',cls:'c12',tbl:1},{id:'o_1',t:'Ortak müşterilerde ARPU değişimi',d:'A ayından B ayına ARPU değişim oranına göre müşteri adedi',cls:'c7'},{id:'o_2',t:'Geçiş matrisi',d:'A ayındaki gruptan B ayında hangi gruba geçildi? (satır toplamı %100)',cls:'c5',h:'k'}]);$('o_grid').dataset.b=1;}
  if(!h){$('o_sum').innerHTML='<p style="color:#5E6E8C">Bu ay çifti için ortak müşteri verisi yok.</p>';return;}
  $('o_sum').innerHTML=`<div class="kpis" style="grid-template-columns:repeat(4,1fr);margin:0">${[{l:'Ortak müşteri',cur:h.n,fmt:fInt,sub:`A portföyünün ${fP1(100*h.n/recT(a).n)}'i`,hi:1},{l:'ARPU\'su artan (>%5)',cur:100*h.up,fmt:fP1,sub:fInt(h.n*h.up)+' müşteri'},{l:'ARPU\'su düşen (<−%5)',cur:100*h.down,fmt:fP1,sub:fInt(h.n*h.down)+' müşteri'},{l:'Medyan ARPU değişimi',cur:100*h.med,fmt:v=>(v>0?'+':'')+nf1.format(v)+'%',sub:'ortalama '+(h.mean>0?'+':'')+nf1.format(100*h.mean)+'%'}].map(kpiHtml).join('')}</div>`;
  const E=META.histEdges,lb=h.h.map((_,i)=>{const lo=E[i],hi=E[i+1];const f=v=>(v>0?'+':v<0?'−':'')+Math.abs(Math.round(v*100));return i===0?`< ${f(hi)}%`:i===h.h.length-1?`> ${f(lo)}%`:`${f(lo)} … ${f(hi)}%`;});
  plot('o_1',[{type:'bar',x:lb,y:h.h,marker:{color:h.h.map((_,i)=>E[i+1]<=-.05?C.bad:E[i]>=.05?C.good:C.cmp)},text:h.h.map(v=>fInt(v)),textposition:'outside',cliponaxis:false,textfont:{size:10.5,color:C.mut},hovertemplate:'%{x}<br><b>%{y:,.0f}</b> müşteri<extra></extra>'}],{showlegend:false,xaxis:{tickangle:-35,tickfont:{size:10.5}},yaxis:{tickformat:',.0f'},margin:{l:56,r:10,t:20,b:80}});
  const dk=S2.oDim,m=PAIR.get(a+'_'+b+'_'+dk);if(!m){$('o_2').innerHTML='<p style="color:#5E6E8C">Bu kırılım için geçiş matrisi hesaplanmadı (tarife, dijital segment, cihaz, sözleşme ve şehir için vardır).</p>';return;}
  const labs=dl(dk),n=labs.length,z=labs.map((_,i)=>{const row=m.slice(i*n,i*n+n),s=sum(row);return row.map(v=>s?100*v/s:NaN);});
  plot('o_2',[{type:'heatmap',z,x:labs.map(l=>'→ '+l),y:labs,colorscale:BLUES,xgap:2,ygap:2,showscale:false,zmin:0,zmax:100,texttemplate:'%{z:.0f}',textfont:{size:11},hovertemplate:'A: %{y}<br>B: %{x}<br><b>%{z:.1f}%</b><extra></extra>'}],{showlegend:false,yaxis:{autorange:'reversed'},xaxis:{side:'top',tickangle:-35,tickfont:{size:10}},margin:{l:130,r:10,t:90,b:6}});}

/* ── İlgi alanı değişimi ── */
const TB3={k:'rel',d:-1};
function p2i(){const a=S2.a,b=S2.b,TA=recT(a),TB=recT(b);if(!$('n_grid').dataset.b){mkCards('n_grid',[{id:'n_1',t:'Müşteri başı arama adedi değişimi (%)',d:'Her ilgi alanı için B ayının A ayına göre değişimi',cls:'c6',st:'height:640px'},{id:'n_2',t:'Arama yapan müşteri payı değişimi (puan)',d:'Ilgi alanında en az 1 arama yapan müşterilerin payı',cls:'c6',st:'height:640px'},{id:'n_t',t:'İlgi alanı tablosu',d:'Başlığa tıklayarak sıralayın',cls:'c12',tbl:1}]);$('n_grid').dataset.b=1;}
  const rows=IV.map((nm,j)=>({nm,aA:intAvg(TA,j),aB:intAvg(TB,j),rel:relp(intAvg(TB,j),intAvg(TA,j)),pA:intPen(TA,j),pB:intPen(TB,j),dp:intPen(TB,j)-intPen(TA,j)}));
  const o1=rows.slice().sort((p,q)=>q.rel-p.rel),o2=rows.slice().sort((p,q)=>q.dp-p.dp);
  plot('n_1',[{type:'bar',orientation:'h',y:o1.map(x=>x.nm),x:o1.map(x=>x.rel),marker:{color:o1.map(x=>x.rel>=0?C.blue:C.cmp)},text:o1.map(x=>(x.rel>0?'+':'')+nf1.format(x.rel)+'%'),textposition:'outside',cliponaxis:false,textfont:{size:10.5,color:C.mut}}],{showlegend:false,yaxis:{autorange:'reversed',automargin:true},xaxis:{gridcolor:C.grid,zeroline:true,zerolinecolor:C.navy,ticksuffix:'%'},margin:{l:110,r:50,t:10,b:30}});
  plot('n_2',[{type:'bar',orientation:'h',y:o2.map(x=>x.nm),x:o2.map(x=>x.dp),marker:{color:o2.map(x=>x.dp>=0?C.blue:C.cmp)},text:o2.map(x=>(x.dp>0?'+':'')+nf2.format(x.dp)),textposition:'outside',cliponaxis:false,textfont:{size:10.5,color:C.mut}}],{showlegend:false,yaxis:{autorange:'reversed',automargin:true},xaxis:{gridcolor:C.grid,zeroline:true,zerolinecolor:C.navy,ticksuffix:' p'},margin:{l:110,r:50,t:10,b:30}});
  sortTable($('n_t'),[['nm','İlgi alanı'],['aA','A: arama/müşteri'],['aB','B: arama/müşteri'],['rel','Değişim %'],['pA','A: arama yapan %'],['pB','B: arama yapan %'],['dp','Fark (puan)']].map(([k,h])=>({k,h,f:r=>k==='nm'?`<b>${r.nm}</b>`:k==='rel'?delta(r.rel,false,v=>nf1.format(v)+'%'):k==='dp'?delta(r.dp,false,v=>nf2.format(v)+' p'):['pA','pB'].includes(k)?fP1(r[k]):nf2.format(r[k])})),rows,TB3,p2i);}

/* ── Bilgilendirme (uyarı motoru) ── */
const fsp=v=>(v>0?'+':v<0?'−':'')+'%'+nf1.format(Math.abs(v)),EXCLM={age:['age'],ten:['ten'],ab:['arpu'],gbb:['gb'],at:['arpur'],comp:['comp'],po3:['po3'],pi12:['pi12'],tar:['price']},NOCNT=new Set(['ab','gbb','at','ten']);
function chg(x,y,k){const mA=mean(x,k),mB=mean(y,k);if(!ok(mA)||!ok(mB))return null;const pp=k==='arpur',rel=pp?mB-mA:relp(mB,mA);return {mA,mB,rel,pp};}
function alerts(){const a=S2.a,b=S2.b,{c:thC,o:thO,m:minN}=S2.al,items=[],TA=recT(a),TB=recT(b);
  const totC=relp(TB.n,TA.n),totM={};MET.forEach(m=>{if(m.ok)totM[m.key]=chg(TA,TB,m.key);});
  const totI=IV.map((_,j)=>relp(TB.i[j]/TB.n,TA.i[j]/TA.n));
  mkGroupList().forEach(g=>{const key=g.di<0?null:DIMS[g.di].key,isT=g.di<0;if(key&&['ab','gbb','at'].includes(key))return;g.labs.forEach((cat,ci)=>{const x=recG(a,g,ci),y=recG(b,g,ci);if(!x||!y||x.n<minN||y.n<minN)return;
    const lbl=isT?'Toplam portföyde':`<span class="tg">${g.name}</span>${cat}`,grp=isT?'':`<span class="tg">${g.name}</span>${cat}`;
    if(S2.al.cnt&&!(key&&NOCNT.has(key))){const rel=relp(y.n,x.n),dev=rel-totC;if(Math.abs(rel)>=thC&&(isT||Math.abs(dev)>=thC/2))items.push({typ:'c',sc:Math.abs(isT?rel:dev)/thC*Math.log10(Math.min(x.n,y.n)),cls:rel>0?'up':'dn',ic:rel>0?'▲':'▼',html:`<b>${isT?'Toplam müşteri':lbl}</b> adedi <b>%${nf1.format(Math.abs(rel))}</b> ${rel>0?'arttı':'düştü'}`,sm:`${fInt(x.n)} → ${fInt(y.n)} müşteri (${fS(y.n-x.n)})${isT?'':` · segment geneli ${fsp(totC)}`}`});}
    if(S2.al.mean)MET.forEach(m=>{if(!m.ok||(key&&(EXCLM[key]||[]).includes(m.key)))return;const c=chg(x,y,m.key);if(!c)return;const share=SHARE.has(m.key),base=m.key==='arpur'?c.rel:(share?relp(c.mB,c.mA):c.rel);if(share&&c.mA<.5)return;
      const lim=m.key==='arpur'?thO/2:thO,tot=totM[m.key],totV=m.key==='arpur'?tot.rel:(share?relp(tot.mB,tot.mA):tot.rel);if(Math.abs(base)<lim)return;if(!isT&&Math.abs(base-totV)<lim/2)return;
      const sA=sdev(x,m.key),sB=sdev(y,m.key),se=Math.sqrt(sA*sA/x.n+sB*sB/y.n),z=se>0?(c.mB-c.mA)/se:0;if(Math.abs(z)<3)return;
      const risk=RISK.has(m.key),txt=m.key==='arpur'?`${nf1.format(Math.abs(base))} puan`:`%${nf1.format(Math.abs(base))}`,sg=m.key==='arpur'?`${totV>0?'+':totV<0?'−':''}${nf1.format(Math.abs(totV))} puan`:fsp(totV);
      items.push({typ:'m',sc:Math.abs(isT?base:base-totV)/lim*Math.log10(Math.min(x.n,y.n)),cls:risk?(base>0?'dn':'up'):(base>0?'up':'dn'),ic:base>0?'▲':'▼',html:`<b>${lbl}</b> ${isT?'':'müşterilerinde '}${m.name} ${share?'oranı':'ortalaması'} <b>${txt}</b> ${base>0?'arttı':'düştü'}`,sm:`${fmtM(m.key,c.mA)} → ${fmtM(m.key,c.mB)}${isT?'':` · segment geneli ${sg}`}`});});
    if(S2.al.int)IV.forEach((nm,j)=>{if(x.i[j]<300||y.i[j]<300)return;const rel=relp(y.i[j]/y.n,x.i[j]/x.n);if(Math.abs(rel)<thO*2||(!isT&&Math.abs(rel-totI[j])<thO))return;items.push({typ:'i',sc:Math.abs(isT?rel:rel-totI[j])/(thO*2)*Math.log10(Math.min(x.i[j],y.i[j])),cls:'nt',ic:rel>0?'▲':'▼',html:`<b>${lbl}</b> ${isT?'':'müşterilerinde '}<b>${nm}</b> araması <b>%${nf1.format(Math.abs(rel))}</b> ${rel>0?'arttı':'azaldı'}`,sm:`${nf2.format(x.i[j]/x.n)} → ${nf2.format(y.i[j]/y.n)} arama / müşteri${isT?'':` · segment geneli ${fsp(totI[j])}`}`});});});});
  const sec=(typ,ttl,lim)=>{const l=items.filter(i=>i.typ===typ).sort((p,q)=>q.sc-p.sc);return l.length?`<div style="font-weight:700;color:var(--navy);margin:14px 0 2px">${ttl} <span class="tag">${l.length}</span></div>`+l.slice(0,lim).map(i=>`<div class="al ${i.cls}"><div class="i">${i.ic}</div><div><div>${i.html}</div><div class="sm">${i.sm}</div></div></div>`).join('')+(l.length>lim?`<div class="sm" style="color:var(--mut);padding:6px 4px">… ve ${l.length-lim} uyarı daha (eşikleri yükselterek daraltabilirsiniz)</div>`:''):'';};
  $('alerts').innerHTML=(items.length?`<div class="scope" style="padding:0 0 4px"><b>${ML(a)}</b> ile <b>${ML(b)}</b> karşılaştırıldı · <b>${items.length}</b> değişim eşiği aştı. Gruplar, segment genelinden de belirgin ayrışıyorsa listelenir.</div>`:'<p style="color:#5E6E8C">Seçilen eşiklerde dikkat çeken değişim bulunamadı.</p>')+sec('c','Adet değişimleri',15)+sec('m','Ortalama değişimleri',25)+sec('i','İlgi alanı değişimleri',10);}

const R2={g:p2g,d:p2d,a:p2a,o:p2o,i:p2i};
function paint2(){resolve2();if(!D2T[S2.tab]){D2T[S2.tab]=1;R2[S2.tab]();}}
function refresh2(){for(const k in R2)D2T[k]=0;paint2();alerts();}
RENDER.p2=()=>{if(!D2T._init){D2T._init=1;refresh2();}else paint2();};
function init2(){
  const fillB=()=>{const o=[];for(let t=S2.a+1;t<NT;t++)o.push([t,ML(t)]);if(S2.b<=S2.a)S2.b=NT-1;sel($('p2b'),o,S2.b);};
  sel($('p2a'),Array.from({length:NT-1},(_,t)=>[t,ML(t)]),S2.a);fillB();sel($('p2d'),DIMS.map((d,i)=>[i,d.name]),S2.d);
  sel($('m_met'),MET.filter(m=>m.ok).map(m=>[m.key,m.name]),S2.met);sel($('o_dim'),DIMS.map((d,i)=>[i,d.name]).filter(x=>['tar','dig','dev','con','city'].includes(DIMS[x[0]].key)),(DI.tar!=null?DI.tar:0));S2.oDim=+$('o_dim').value;
  $('p2a').onchange=e=>{S2.a=+e.target.value;fillB();refresh2();};$('p2b').onchange=e=>{S2.b=+e.target.value;refresh2();};$('p2d').onchange=e=>{S2.d=+e.target.value;D2T.d=D2T.a=0;paint2();};
  $('m_met').onchange=e=>{S2.met=e.target.value;p2a();};$('o_dim').onchange=e=>{S2.oDim=+e.target.value;p2o();};
  document.querySelectorAll('#p2tabs button').forEach(b=>b.onclick=()=>{S2.tab=b.dataset.t;document.querySelectorAll('#p2tabs button').forEach(x=>x.classList.toggle('on',x===b));document.querySelectorAll('#p2 .tab').forEach(s=>s.classList.toggle('on',s.id==='p2-'+S2.tab));paint2();
    setTimeout(()=>document.querySelectorAll('#p2-'+S2.tab+' .ch').forEach(e=>{try{Plotly.Plots.resize(e)}catch(_){}}),60);});
  const sl=(id,key,fmt)=>{const el=$(id);el.value=S2.al[key];$(id+'_v').textContent=fmt(S2.al[key]);el.oninput=()=>{S2.al[key]=+el.value;$(id+'_v').textContent=fmt(S2.al[key]);alerts();};};
  sl('al_a','c',v=>'%'+v);sl('al_o','o',v=>'%'+v);sl('al_m','m',v=>fInt(v));
  [['al_c','cnt'],['al_mean','mean'],['al_i','int']].forEach(([id,k])=>$(id).onchange=e=>{S2.al[k]=e.target.checked;alerts();});}
init2();

/* ═══════════════ SAYFA 3 · MÜŞTERİ HAREKETLERİ ═══════════════ */
const EVL=META.eventLabels,EVN=META.events,EVC=[C.navy,C.blue,C.sky,'#13A97B','#6FD3B0',C.cmp,'#7B6CC4','#A99BDD',C.bad,C.mut,'#C9C3D6'];
const EVK=new Map();{const E=D2.ev;for(let i=0;i<E.t.length;i++)EVK.set(((E.t[i]*16+E.e[i])*32+E.d[i])*16+E.c[i],{n:E.n[i],s:E.s[i]});}
const EVIK=new Map();{const E=D2.evi;for(let i=0;i<E.t.length;i++)EVIK.set(E.t[i]*16+E.e[i],{n:E.n[i],s:E.s[i],i:E.i[i],u:E.u[i]});}
const GROUPS=[['in_all','Tüm girişler',[1,2,3,4,5]],['in_int',"Şirket içi girişler (Prepaid + Postpaid'den)",[3,4]],['in_ext','Şirket dışı girişler (yeni satış + numara taşıma)',[1,2]],['out_all','Tüm çıkışlar',[6,7,8,9,10]],['out_int',"Şirket içi çıkışlar (Prepaid + Postpaid'e)",[6,7]],['out_ext','Şirket dışı çıkışlar (rakip + hat kapatma + dosyadan kaybolan)',[8,9,10]]]
  .concat([1,2,3,4,5,6,7,8,9,10].map(e=>['e'+e,EVL[e],[e]]));
const GM=new Map(GROUPS.map(g=>[g[0],g]));
const S3={p:'all',d:0,g:'out_all',met:'arpu',tab:'g'},D3T={};
const evr=(t,e,d,c)=>EVK.get(((t*16+e)*32+d)*16+c);
const ts3=()=>S3.p==='all'?Array.from({length:NT-1},(_,i)=>i+1):[+S3.p];
function agg(ts,es,d,c){const o={n:0,s:new Float64Array(MET.length)};for(const t of ts)for(const e of es){const r=evr(t,e,d,c);if(r){o.n+=r.n;for(let j=0;j<MET.length;j++)o.s[j]+=r.s[j];}}return o;}
function aggStock(ts,d,c,off=0){const o={n:0,s:new Float64Array(MET.length)};for(const t0 of ts){const t=t0+off,r=d===0?recT(t):rec(t,d-1,c);if(r){o.n+=r.n;for(let j=0;j<MET.length;j++)o.s[j]+=r.s[j];}}return o;}
const mA=(o,k)=>o.n?(SHARE.has(k)?100:1)*o.s[MJ[k]]/o.n:NaN;
function aggInt(ts,es){const o={n:0,i:new Array(NI).fill(0)};for(const t of ts)for(const e of es){const r=EVIK.get(t*16+e);if(r){o.n+=r.n;r.i.forEach((v,j)=>o.i[j]+=v);}}return o;}
function stockInt(ts){const o={n:0,i:new Array(NI).fill(0)};for(const t of ts){const r=recT(t);o.n+=r.n;r.i.forEach((v,j)=>o.i[j]+=v);}return o;}
const evCount=(ts,es)=>agg(ts,es,0,0).n;
function resolve3(){const g=GM.get(S3.g);$('p3s').innerHTML=`Dönem: <b>${S3.p==='all'?`${MS(1)} – ${MS(NT-1)} (tüm geçişler)`:ML(+S3.p)}</b> · Kırılım: <b>${S3.d===0?'—':dnm(S3.d-1)}</b> · Hareket grubu: <b>${g[1]}</b>`;}

/* ── Genel akış ── */
function p3g(){const ts=ts3(),all=Array.from({length:NT-1},(_,i)=>i+1);if(!$('h_grid').dataset.b){mkCards('h_grid',[{id:'h_flow',t:'Aylık giriş ve çıkışlar',d:'Pozitif: portföye girenler · negatif: çıkanlar · çizgi: net',cls:'c12'},{id:'h_sk',t:'Seçili dönemde müşteri akışı',d:'Kaynak → portföy → hedef · şerit kalınlığı müşteri sayısıdır',cls:'c7',h:'k'},{id:'h_tbl',t:'Aylık hareket tablosu',d:'',cls:'c5',tbl:1}]);$('h_grid').dataset.b=1;}
  const n=es=>evCount(ts,es),inInt=n([3,4]),inExt=n([1,2]),inOth=n([5]),outInt=n([6,7]),outExt=n([8,9,10]),net=inInt+inExt+inOth-outInt-outExt,t0=ts[0],t1=ts[ts.length-1];
  $('h_kpis').innerHTML=[{l:'Şirket içi giriş',cur:inInt,fmt:fInt,sub:"Prepaid/Postpaid'den",hi:1},{l:'Şirket dışı giriş',cur:inExt,fmt:fInt,sub:`${fInt(n([1]))} yeni satış · ${fInt(n([2]))} numara taşıma`,hi:1},{l:'Şirket içi çıkış',cur:outInt,fmt:fInt,sub:"Prepaid/Postpaid'e"},{l:'Şirket dışı çıkış',cur:outExt,fmt:fInt,sub:`${fInt(n([8]))} rakibe · ${fInt(n([9]))} kapatma · ${fInt(n([10]))} dosyadan kaybolan`},
    {l:'Net değişim',cur:net,fmt:fS,sub:`${fInt(recT(t0-1).n)} → ${fInt(recT(t1).n)}`,hi:1},{l:'Sınıflanamayan giriş',cur:inOth,fmt:fInt,sub:'bayrağı olmayan yeni müşteri'}].map(kpiHtml).join('');
  const X=all.map(MS);plot('h_flow',[1,2,3,4,5].map(e=>({type:'bar',x:X,y:all.map(t=>agg([t],[e],0,0).n),name:EVL[e],marker:{color:EVC[e]},hovertemplate:'%{x}<br>'+EVL[e]+': <b>%{y:,.0f}</b><extra></extra>'})).concat([6,7,8,9,10].map(e=>({type:'bar',x:X,y:all.map(t=>-agg([t],[e],0,0).n),name:EVL[e],marker:{color:EVC[e]},customdata:all.map(t=>agg([t],[e],0,0).n),hovertemplate:'%{x}<br>'+EVL[e]+': <b>%{customdata:,.0f}</b><extra></extra>'}))).concat([
    {type:'scatter',mode:'lines+markers',x:X,y:all.map(t=>recT(t).n-recT(t-1).n),name:'Net',line:{color:C.navy,width:2.6},marker:{size:6}}]),{barmode:'relative',bargap:.3,yaxis:{tickformat:',.0f',zeroline:true,zerolinecolor:C.navy},legend:{font:{size:10.5}},margin:{l:58,r:10,t:50,b:40}});
  const srcs=[[1,0],[2,1],[3,2],[4,3],[5,4]],dst=[[6,5],[7,6],[8,7],[9,8],[10,9]],lab=[...[1,2,3,4,5].map(e=>EVL[e]),'Portföy',...[6,7,8,9,10].map(e=>EVL[e])],col=[...[1,2,3,4,5].map(e=>EVC[e]),C.blue,...[6,7,8,9,10].map(e=>EVC[e])];
  const sv=[],tv=[],vv=[],lc=[];[1,2,3,4,5].forEach((e,i)=>{const v=n([e]);if(v){sv.push(i);tv.push(5);vv.push(v);lc.push(rgba(EVC[e],.45));}});[6,7,8,9,10].forEach((e,i)=>{const v=n([e]);if(v){sv.push(5);tv.push(6+i);vv.push(v);lc.push(rgba(EVC[e],.45));}});
  Plotly.react('h_sk',[{type:'sankey',arrangement:'snap',valueformat:',.0f',node:{pad:14,thickness:16,line:{width:0},label:lab,color:col},link:{source:sv,target:tv,value:vv,color:lc},textfont:{family:FONT,size:11.5,color:C.text}}],{paper_bgcolor:'rgba(0,0,0,0)',font:{family:FONT,color:C.text},margin:{l:6,r:6,t:6,b:6}},CFG);
  const rows=all.map(t=>({m:MS(t),open:recT(t-1).n,ei:agg([t],[1,2],0,0).n,ii:agg([t],[3,4],0,0).n,oi:agg([t],[6,7],0,0).n,oe:agg([t],[8,9,10],0,0).n,x:agg([t],[5],0,0).n,close:recT(t).n}));
  $('h_tbl').innerHTML='<table><thead><tr>'+['Ay','Açılış','Dış giriş','İç giriş','İç çıkış','Dış çıkış','Diğer*','Kapanış'].map(h=>`<th style="cursor:default">${h}</th>`).join('')+'</tr></thead><tbody>'+rows.map(r=>`<tr><td><b>${r.m}</b></td><td>${fInt(r.open)}</td><td>${fInt(r.ei)}</td><td>${fInt(r.ii)}</td><td>${fInt(r.oi)}</td><td>${fInt(r.oe)}</td><td class="z">${fInt(r.x)}</td><td><b>${fInt(r.close)}</b></td></tr>`).join('')+'</tbody></table><div class="d" style="margin-top:8px">* bayrağı olmayan (sınıflanamayan) giriş</div>';}

/* ── Hareket profili ── */
function p3p(){const ts=ts3();if(!$('hp_grid').dataset.b){mkCards('hp_grid',[{id:'hp_tbl',t:'Hareket türüne göre profil',d:'Her satır bir hareket türü; değerler o müşterilerin ortalamasıdır. Kırmızı: portföyden %15+ yüksek · mavi: %15+ düşük',cls:'c12',tbl:1},{id:'hp_1',t:'Seçili ölçüde hareket türleri',d:'Çizgi: portföy ortalaması',cls:'c6'},{id:'hp_2',t:"Hareket gruplarının portföye göre indeksi",d:'100 = portföy ortalaması',cls:'c6'}]);$('hp_grid').dataset.b=1;}
  const base=aggStock(ts,0,0),ms=MET.filter(m=>m.ok).map(m=>m.key);
  const rows=[{nm:'Portföy (baz)',o:base,base:true}].concat([1,2,3,4,5,6,7,8,9,10].map(e=>({nm:EVL[e],o:agg(ts,[e],0,0),e})).filter(r=>r.o.n>0));
  $('hp_tbl').innerHTML='<table><thead><tr><th style="cursor:default">Hareket türü</th><th style="cursor:default">Adet</th>'+ms.map(k=>`<th style="cursor:default">${MET[MJ[k]].name}</th>`).join('')+'</tr></thead><tbody>'+rows.map(r=>`<tr><td><b>${r.nm}</b></td><td>${fInt(r.o.n)}</td>`+ms.map(k=>{const v=mA(r.o,k),b=mA(base,k),ix=b?v/b:1;const cls=!r.base&&k!=='arpur'&&ok(ix)&&b>0?(ix>=1.15?'hi':ix<=.85?'lo':''):'';return `<td class="${cls}">${fmtM(k,v)}</td>`;}).join('')+'</tr>').join('')+'</tbody></table>';
  const k=S3.met,ev=rows.filter(r=>!r.base);
  plot('hp_1',[{type:'bar',orientation:'h',y:ev.map(r=>r.nm),x:ev.map(r=>mA(r.o,k)),marker:{color:ev.map(r=>EVC[r.e])},text:ev.map(r=>fmtM(k,mA(r.o,k))),textposition:'outside',cliponaxis:false,textfont:{size:11,color:C.mut},hovertemplate:'%{y}<br><b>%{x:.2f}</b><extra></extra>'}],
    {showlegend:false,yaxis:{autorange:'reversed',automargin:true},xaxis:{gridcolor:C.grid,rangemode:'tozero'},shapes:[{type:'line',yref:'paper',y0:0,y1:1,x0:mA(base,k),x1:mA(base,k),line:{color:C.yellow,width:3,dash:'dash'}}],margin:{l:230,r:70,t:10,b:30}});
  const gs=[['Şirket içi giriş',[3,4],C.good],['Şirket dışı giriş',[1,2],C.blue],['Şirket içi çıkış',[6,7],'#7B6CC4'],['Şirket dışı çıkış',[8,9,10],C.bad]];
  plot('hp_2',gs.map(g=>{const o=agg(ts,g[1],0,0);return {type:'bar',name:g[0],x:ms.filter(x=>x!=='arpur').map(x=>MET[MJ[x]].name),y:ms.filter(x=>x!=='arpur').map(x=>100*mA(o,x)/mA(base,x)),marker:{color:g[2]},hovertemplate:'%{x}<br>'+g[0]+' <b>%{y:.0f}</b><extra></extra>'};}),
    {barmode:'group',bargap:.25,yaxis:{rangemode:'tozero',title:{text:'indeks'}},xaxis:{tickfont:{size:10},tickangle:-25},shapes:[{type:'line',xref:'paper',x0:0,x1:1,y0:100,y1:100,line:{color:C.navy,width:2,dash:'dot'}}],margin:{l:56,r:10,t:40,b:90}});}

/* ── Kırılımlarda hareket ── */
const TB4={k:'rate',d:-1};
function p3k(){const ts=ts3(),g=GM.get(S3.g),es=g[2],d=S3.d,isIn=g[0].startsWith('in')||(g[0][0]==='e'&&+g[0].slice(1)<=5);
  if(d===0){$('hk_grid').innerHTML='<div class="note" style="grid-column:1/-1">Bu sekme için üstten bir <b>Kırılım</b> seçin (cinsiyet, yaş, tarife, cihaz vb.).</div>';$('hk_grid').dataset.b='';return;}
  const di=d-1,labs=dl(di);if(!$('hk_grid').dataset.b){mkCards('hk_grid',[{id:'hk_1',t:'Hareket edenlerin dağılımı ve portföy',d:'Hareket eden müşterilerin kırılım payı ile portföyün payı',cls:'c6'},{id:'hk_2',t:'Hareket oranı',d:isIn?'Gelen müşteri ÷ açılış portföyü (%)':'Giden müşteri ÷ açılış portföyü (%)',cls:'c6'},
    {id:'hk_3',t:'Hareket edenlerin ARPU\'su ve portföy ARPU\'su',d:'₺',cls:'c6'},{id:'hk_4',t:'Hareket türü × grup',d:'Her hareket türü içinde grupların payı (satır toplamı %100)',cls:'c6'},{id:'hk_t',t:'Detay tablosu',d:'Başlığa tıklayarak sıralayın',cls:'c12',tbl:1}]);$('hk_grid').dataset.b=1;}
  const tot=agg(ts,es,0,0),sb=aggStock(ts,0,0),openT=aggStock(ts,0,0,-1);
  const rows=labs.map((nm,c)=>{const m=agg(ts,es,d,c),p=aggStock(ts,d,c),op=aggStock(ts,d,c,-1);return {nm,n:m.n,pn:p.n,rate:op.n?100*m.n/op.n:NaN,ms:tot.n?100*m.n/tot.n:NaN,ps:sb.n?100*p.n/sb.n:NaN,ix:(tot.n&&sb.n&&p.n)?(m.n/tot.n)/(p.n/sb.n)*100:NaN,am:mA(m,'arpu'),ap:mA(p,'arpu'),gm:mA(m,'gb'),gp:mA(p,'gb')};});
  const h=labs.length>6;
  plot('hk_1',[{type:'bar',name:'Portföy',[h?'y':'x']:labs,[h?'x':'y']:rows.map(r=>r.ps),orientation:h?'h':'v',marker:{color:C.cmp}},{type:'bar',name:'Hareket edenler',[h?'y':'x']:labs,[h?'x':'y']:rows.map(r=>r.ms),orientation:h?'h':'v',marker:{color:C.blue},text:rows.map(r=>fP1(r.ms)),textposition:'outside',cliponaxis:false,textfont:{size:10.5,color:C.mut}}],
    Object.assign({barmode:'group',bargap:.3},h?{yaxis:{autorange:'reversed',automargin:true},xaxis:{ticksuffix:'%',gridcolor:C.grid},margin:{l:130,r:50,t:34,b:30}}:{yaxis:{ticksuffix:'%',rangemode:'tozero'},xaxis:{tickfont:{size:10.5}},margin:{l:46,r:10,t:34,b:labs.some(l=>l.length>9)?72:40}}));
  const avgRate=openT.n?100*tot.n/openT.n:NaN;
  plot('hk_2',[{type:'bar',[h?'y':'x']:labs,[h?'x':'y']:rows.map(r=>r.rate),orientation:h?'h':'v',marker:{color:rows.map(r=>r.rate>avgRate*1.1?(isIn?C.good:C.bad):C.blue)},text:rows.map(r=>fP2(r.rate)),textposition:'outside',cliponaxis:false,textfont:{size:10.5,color:C.mut}}],
    Object.assign({showlegend:false},h?{yaxis:{autorange:'reversed',automargin:true},xaxis:{ticksuffix:'%',gridcolor:C.grid,rangemode:'tozero'},shapes:[{type:'line',yref:'paper',y0:0,y1:1,x0:avgRate,x1:avgRate,line:{color:C.yellow,width:3,dash:'dash'}}],margin:{l:130,r:60,t:10,b:30}}:{yaxis:{ticksuffix:'%',rangemode:'tozero'},xaxis:{tickfont:{size:10.5}},shapes:[{type:'line',xref:'paper',x0:0,x1:1,y0:avgRate,y1:avgRate,line:{color:C.yellow,width:3,dash:'dash'}}],margin:{l:50,r:10,t:20,b:labs.some(l=>l.length>9)?72:40}}));
  plot('hk_3',[{type:'bar',name:'Portföy',x:labs,y:rows.map(r=>r.ap),marker:{color:C.cmp}},{type:'bar',name:'Hareket edenler',x:labs,y:rows.map(r=>r.am),marker:{color:C.blue},text:rows.map(r=>fTL(r.am)),textposition:'outside',cliponaxis:false,textfont:{size:10.5,color:C.mut}}],{barmode:'group',bargap:.3,yaxis:{tickprefix:'₺',rangemode:'tozero'},xaxis:{tickfont:{size:10.5}},margin:{l:56,r:10,t:34,b:labs.some(l=>l.length>9)?72:40}});
  const evs=[1,2,3,4,5,6,7,8,9,10].filter(e=>agg(ts,[e],0,0).n>0),z=evs.map(e=>{const t=agg(ts,[e],0,0).n;return labs.map((_,c)=>t?100*agg(ts,[e],d,c).n/t:NaN);});
  plot('hk_4',[{type:'heatmap',z,x:labs,y:evs.map(e=>EVL[e]),colorscale:BLUES,xgap:2,ygap:2,showscale:false,texttemplate:'%{z:.0f}',textfont:{size:10.5},hovertemplate:'%{y}<br>%{x}<br><b>%{z:.1f}%</b><extra></extra>'}],{showlegend:false,yaxis:{autorange:'reversed',automargin:true},xaxis:{side:'top',tickangle:-30,tickfont:{size:10}},margin:{l:210,r:10,t:80,b:6}});
  sortTable($('hk_t'),[['nm','Grup'],['pn','Portföy (dönem toplamı)'],['n','Hareket eden'],['rate','Hareket oranı'],['ms','Hareket payı'],['ps','Portföy payı'],['ix','Pay indeksi'],['am','Hareket eden ARPU'],['ap','Portföy ARPU'],['gm','Hareket eden GB'],['gp','Portföy GB']].map(([k,hh])=>({k,h:hh,
    f:r=>k==='nm'?`<b>${r.nm}</b>`:['pn','n'].includes(k)?fInt(r[k]):['rate'].includes(k)?fP2(r[k]):['ms','ps'].includes(k)?fP1(r[k]):k==='ix'?`<span class="${r.ix>115?'dn':r.ix<85?'up':''}">${ok(r.ix)?nf0.format(r.ix):'–'}</span>`:['am','ap'].includes(k)?fTL(r[k]):nf1.format(r[k])})),rows,TB4,p3k);}

/* ── İlgi alanları ── */
function p3i(){const ts=ts3(),g=GM.get(S3.g);if(!$('hi_grid').dataset.b){mkCards('hi_grid',[{id:'hi_1',t:'Seçili hareket grubunun ilgi alanı indeksi',d:'Hareket edenlerin müşteri başı arama adedi ÷ portföy · 100 = portföy ortalaması',cls:'c5',st:'height:700px'},{id:'hi_2',t:'Hareket grupları × ilgi alanı indeksi',d:'100 = portföy ortalaması',cls:'c7',st:'height:700px'}]);$('hi_grid').dataset.b=1;}
  const base=stockInt(ts),bi=j=>base.i[j]/base.n,ix=(o,j)=>o.n&&bi(j)>0?100*(o.i[j]/o.n)/bi(j):NaN,o=aggInt(ts,g[2]);
  if(!o.n){$('hi_1').innerHTML='<p style="color:#5E6E8C">Bu dönemde seçili grupta müşteri yok.</p>';}else{
    const ord=IV.map((_,j)=>[ix(o,j),j]).filter(x=>ok(x[0])).sort((a,b)=>b[0]-a[0]);
    plot('hi_1',[{type:'bar',orientation:'h',y:ord.map(x=>IV[x[1]]),x:ord.map(x=>x[0]-100),base:100,marker:{color:ord.map(x=>x[0]>=100?C.bad:C.blue)},text:ord.map(x=>nf0.format(x[0])),textposition:'outside',cliponaxis:false,textfont:{size:11,color:C.mut},customdata:ord.map(x=>[o.i[x[1]]/o.n,bi(x[1])]),hovertemplate:'%{y}<br>İndeks <b>%{text}</b><br>Hareket eden: %{customdata[0]:.2f} · portföy: %{customdata[1]:.2f} arama/müşteri<extra></extra>'}],
      {showlegend:false,yaxis:{autorange:'reversed',automargin:true},xaxis:{gridcolor:C.grid,zeroline:true,zerolinecolor:C.navy,tickvals:[-50,0,50,100],ticktext:['50','100','150','200'],range:[Math.min(40,ord[ord.length-1][0]-10)-100,Math.max(ord[0][0]+15,140)-100]},margin:{l:110,r:50,t:10,b:30}});}
  const gs=[['in_int','Şirket içi giriş'],['in_ext','Şirket dışı giriş'],['out_int','Şirket içi çıkış'],['out_ext','Şirket dışı çıkış']],ord2=IV.map((_,j)=>j);
  plot('hi_2',[{type:'heatmap',z:ord2.map(j=>gs.map(([k])=>{const oo=aggInt(ts,GM.get(k)[2]);return ix(oo,j);})),x:gs.map(x=>x[1]),y:ord2.map(j=>IV[j]),zmin:40,zmax:200,colorscale:[[0,'#4DA3FF'],[.3,'#EAF2FF'],[.45,'#FFFFFF'],[.65,'#FFE08A'],[1,'#E5484D']],xgap:2,ygap:2,showscale:true,colorbar:{thickness:10,len:.6,tickfont:{size:10,color:C.mut}},texttemplate:'%{z:.0f}',textfont:{size:11}}],{showlegend:false,yaxis:{autorange:'reversed'},xaxis:{side:'top',tickfont:{size:11}},margin:{l:110,r:10,t:50,b:6}});}

const R3={g:p3g,p:p3p,k:p3k,i:p3i};
function paint3(){resolve3();if(!D3T[S3.tab]){D3T[S3.tab]=1;R3[S3.tab]();}}
function refresh3(){for(const k in R3)D3T[k]=0;paint3();}
RENDER.p3=()=>{if(!D3T._init){D3T._init=1;refresh3();}else paint3();};
function init3(){
  sel($('p3p'),[['all','Tüm dönem (tüm geçişler)'],...Array.from({length:NT-1},(_,i)=>[i+1,`${ML(i)} → ${ML(i+1)}`])],S3.p);
  sel($('p3d'),[[0,'— (kırılım yok)'],...DIMS.map((d,i)=>[i+1,d.name])],S3.d);sel($('p3e'),GROUPS.map(g=>[g[0],g[1]]),S3.g);sel($('h_met'),MET.filter(m=>m.ok).map(m=>[m.key,m.name]),S3.met);
  $('p3p').onchange=e=>{S3.p=e.target.value;refresh3();};$('p3d').onchange=e=>{S3.d=+e.target.value;$('hk_grid').dataset.b='';D3T.k=0;paint3();};$('p3e').onchange=e=>{S3.g=e.target.value;D3T.k=D3T.i=0;paint3();};$('h_met').onchange=e=>{S3.met=e.target.value;p3p();};
  document.querySelectorAll('#p3tabs button').forEach(b=>b.onclick=()=>{S3.tab=b.dataset.t;document.querySelectorAll('#p3tabs button').forEach(x=>x.classList.toggle('on',x===b));document.querySelectorAll('#p3 .tab').forEach(s=>s.classList.toggle('on',s.id==='p3-'+S3.tab));paint3();
    setTimeout(()=>document.querySelectorAll('#p3-'+S3.tab+' .ch').forEach(e=>{try{Plotly.Plots.resize(e)}catch(_){}}),60);});}
init3();
/* ═══════════════ başlat ═══════════════ */
$('load').style.display='none';
const h0=location.hash.replace('#','');setPage(['p1','p2','p3'].includes(h0)?h0:'p1');
let _rt;window.addEventListener('resize',()=>{clearTimeout(_rt);_rt=setTimeout(()=>document.querySelectorAll('.page.on .ch').forEach(e=>{try{Plotly.Plots.resize(e)}catch(_){}}),150);});
})();

</script>
</body>
</html>
'''

### 8 · Oluştur

In [ ]:
# ───────────── HTML ÜRET ─────────────
import html as _html

def plotly_tag():
    if PLOTLY_GOMULU:
        try:
            from plotly.offline import get_plotlyjs
            return "<script>" + get_plotlyjs() + "</script>"
        except Exception as e:
            print("⚠ plotly paketi bulunamadı, CDN kullanılacak:", e)
    return '<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>'

def html_uret(yol=CIKTI_HTML):
    out = (HTML_SABLON.replace('__PLOTLY__', plotly_tag()).replace('__TITLE__', f"{SEGMENT_ADI} · Segment Raporu")
                      .replace('__ENC__', 'gz' if VERI_SIKISTIR else 'json').replace('__DATA1__', PAYLOAD[0]).replace('__DATA2__', PAYLOAD[1]))
    with open(yol, 'w', encoding='utf-8') as f:
        f.write(out)
    return yol

yol = html_uret()
print(f"✔ {yol} oluşturuldu ({os.path.getsize(yol)/1e6:.1f} MB) — tek dosya, e-postayla paylaşılabilir.")
from IPython.display import HTML, FileLink, display
display(FileLink(yol, result_html_prefix="Tam ekran aç → "))
if NOTEBOOKTA_GOSTER:
    with open(yol, encoding='utf-8') as f:
        display(HTML('<iframe srcdoc="' + _html.escape(f.read(), quote=True) + '" width="100%" height="1000" style="border:0"></iframe>'))